# League of Legends ranked dashboards

Pulls your ranked games from the Riot API (and, optionally, OP.GG) and shows two interactive dashboards right here in the notebook:

* **Ranked review**: record, rank, what wins and loses your games, gold and deaths minute by minute, objectives, and a short list of highlights worked out from your own games.
* **Champions & matchups**: click any champion you played (or faced) to see your stats, every matchup, and a gold timeline of each game with your kills, deaths, assists and every objective marked.

Both dashboards are also saved as standalone `.html` files you can download from the Files panel and open in any browser.

**Before you start**

1. Get a development API key at [developer.riotgames.com](https://developer.riotgames.com) (it expires every 24 hours).
2. In Colab, open **Secrets** (the key icon on the left), add `RIOT_API_KEY` with your key, and turn on *Notebook access*.
3. Put your Riot ID in the settings cell below, then **Runtime → Run all**.

The Riot pull takes about 2 minutes per 50 games because of the development key's rate limit.

In [ ]:
# ==== Settings: change these to your account ====
GAME_NAME = "YourGameName"      # your Riot ID: the part before the #
TAG_LINE = "NA1"                # the part after the #
REGION = "NA"                   # NA, EUW, EUNE, KR, JP, BR, LAN, LAS, OCE, TR, RU
MATCHES_TO_PULL = 200           # how many recent games to pull
RANKED_SOLO_ONLY = True         # Riot pull: only ranked solo/duo games
TIMEZONE = "America/New_York"   # your time zone, for dates and start times
DATA_DIR = "/content"           # where export files are written and read

import os
assert GAME_NAME != "YourGameName", "Set GAME_NAME and TAG_LINE above to your Riot ID first."
RIOT_ROUTING = {"NA": "americas", "BR": "americas", "LAN": "americas", "LAS": "americas", "EUW": "europe", "EUNE": "europe",
                "TR": "europe", "RU": "europe", "KR": "asia", "JP": "asia", "OCE": "sea"}[REGION]
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)
print(f"Ready for {GAME_NAME}#{TAG_LINE} ({REGION})")

## Step 1: pull your games from the Riot API

Writes `match_players_summary.csv`, `all_players_minute_frames.csv`, `all_matches_events_log.csv` and a compressed raw backup to `DATA_DIR`. Skip this cell if those files are already there.

In [ ]:
import os
import time
import urllib.parse
from collections import deque
import json
import gzip
import requests
import pandas as pd

# Your Riot API key comes from Colab's Secrets panel (key icon on the left): add RIOT_API_KEY and turn on "Notebook access".
# Development keys from developer.riotgames.com expire every 24 hours.
try:
    from google.colab import userdata
    RIOT_API_KEY = userdata.get("RIOT_API_KEY")
except Exception:
    RIOT_API_KEY = os.environ.get("RIOT_API_KEY", "")
assert RIOT_API_KEY, "Add your Riot API key as the RIOT_API_KEY secret first."

ROUTING_REGION = RIOT_ROUTING
QUEUE_ID = 420 if RANKED_SOLO_ONLY else None  # 420 = Ranked Solo/Duo

HEADERS = {"X-Riot-Token": RIOT_API_KEY}

# ===========================================================================
# 2. RATE LIMITER (20 req / 1 sec & 100 req / 120 sec) + retrying GET
# ===========================================================================
class DualRateLimiter:
    def __init__(self, short_limit=20, short_sec=1.0, long_limit=100, long_sec=120.0):
        self.short_limit, self.short_sec = short_limit, short_sec
        self.long_limit, self.long_sec = long_limit, long_sec
        self.short_timestamps, self.long_timestamps = deque(), deque()

    def wait(self):
        while True:
            now = time.time()
            while self.short_timestamps and now - self.short_timestamps[0] >= self.short_sec:
                self.short_timestamps.popleft()
            while self.long_timestamps and now - self.long_timestamps[0] >= self.long_sec:
                self.long_timestamps.popleft()
            sleep_time = 0.0
            if len(self.short_timestamps) >= self.short_limit:
                sleep_time = max(sleep_time, self.short_sec - (now - self.short_timestamps[0]))
            if len(self.long_timestamps) >= self.long_limit:
                sleep_time = max(sleep_time, self.long_sec - (now - self.long_timestamps[0]))
            if sleep_time > 0:
                print(f"⏳ Rate limit reached. Pausing {round(sleep_time + 0.1, 1)}s...")
                time.sleep(sleep_time + 0.1)
            else:
                now = time.time()
                self.short_timestamps.append(now)
                self.long_timestamps.append(now)
                break

rate_limiter = DualRateLimiter()

def riot_get(url, retries=4):
    """GET with the rate limiter; waits out 429s and retries 5xx errors."""
    for attempt in range(retries + 1):
        rate_limiter.wait()
        res = requests.get(url, headers=HEADERS, timeout=30)
        if res.status_code == 200:
            return res.json()
        if res.status_code == 429:
            wait = int(res.headers.get("Retry-After", 10))
            print(f"⏳ 429 from Riot. Waiting {wait}s...")
            time.sleep(wait)
            continue
        if res.status_code >= 500 and attempt < retries:
            time.sleep(2 * (attempt + 1))
            continue
        print(f"⚠️ {res.status_code} for {url}: {res.text[:200]}")
        return None
    return None

# ===========================================================================
# 3. GET PUUID & RIOT MATCH IDs
# ===========================================================================
print(f"Resolving account PUUID for {GAME_NAME}#{TAG_LINE}...")
acct = riot_get(f"https://{ROUTING_REGION}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{urllib.parse.quote(GAME_NAME)}/{urllib.parse.quote(TAG_LINE)}")
if not acct:
    raise Exception("Account lookup failed (check the API key and Riot ID).")
puuid = acct["puuid"]
print(f"✅ Found PUUID: {puuid[:16]}...")

print(f"Fetching up to {MATCHES_TO_PULL} Riot Match IDs...")
queue_param = f"&queue={QUEUE_ID}" if QUEUE_ID else ""
riot_match_ids = []
for start_idx in range(0, MATCHES_TO_PULL, 100):
    count = min(100, MATCHES_TO_PULL - start_idx)
    batch = riot_get(f"https://{ROUTING_REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
                     f"?start={start_idx}&count={count}{queue_param}")
    if not batch:
        break
    riot_match_ids.extend(batch)
    if len(batch) < count:
        break
print(f"✅ Retrieved {len(riot_match_ids)} Match IDs.\n")

# ===========================================================================
# 4. HARVEST MATCH DETAILS + TIMELINES
# ===========================================================================
players_rows = []   # one row per player per game (champion, team, role, result, end stats)
frames_rows = []    # one row per player per minute
events_rows = []    # one row per game event
raw_archive = []    # {"match": ..., "timeline": ...} per game

start_time = time.time()
print(f"🚀 Downloading details + timelines for {len(riot_match_ids)} matches (2 calls each)...")

for idx, match_id in enumerate(riot_match_ids, start=1):
    m_data = riot_get(f"https://{ROUTING_REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}")
    t_data = riot_get(f"https://{ROUTING_REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}/timeline")
    if not m_data or not t_data:
        print(f"⚠️ Skipping {match_id} (missing details or timeline)")
        continue
    raw_archive.append({"match": m_data, "timeline": t_data})

    # --- 1. Per-player summary (from match details) ---
    m_info = m_data.get("info", {})
    teams = {t.get("teamId"): t for t in m_info.get("teams", [])}
    for p in m_info.get("participants", []):
        objectives = teams.get(p.get("teamId"), {}).get("objectives", {})
        players_rows.append({
            "match_id": match_id,
            "game_start_utc": pd.to_datetime(m_info.get("gameStartTimestamp"), unit="ms", utc=True),
            "game_duration_s": m_info.get("gameDuration"),
            "queue_id": m_info.get("queueId"),
            "patch": m_info.get("gameVersion"),
            "participant_id": p.get("participantId"),
            "is_target_player": p.get("puuid") == puuid,
            "riot_id": f"{p.get('riotIdGameName')}#{p.get('riotIdTagline')}",
            "team_id": p.get("teamId"),
            "win": p.get("win"),
            "champion": p.get("championName"),
            "position": p.get("teamPosition"),
            "kills": p.get("kills"), "deaths": p.get("deaths"), "assists": p.get("assists"),
            "gold_earned": p.get("goldEarned"),
            "total_cs": (p.get("totalMinionsKilled", 0) + p.get("neutralMinionsKilled", 0)),
            "dmg_to_champs": p.get("totalDamageDealtToChampions"),
            "dmg_taken": p.get("totalDamageTaken"),
            "vision_score": p.get("visionScore"),
            "wards_placed": p.get("wardsPlaced"), "wards_killed": p.get("wardsKilled"),
            "control_wards_bought": p.get("visionWardsBoughtInGame"),
            "time_dead_s": p.get("totalTimeSpentDead"),
            "first_blood_kill": p.get("firstBloodKill"), "first_blood_assist": p.get("firstBloodAssist"),
            "first_tower_kill": p.get("firstTowerKill"),
            "team_first_tower": objectives.get("tower", {}).get("first"),
            "team_first_dragon": objectives.get("dragon", {}).get("first"),
            "team_first_baron": objectives.get("baron", {}).get("first"),
            "team_dragons": objectives.get("dragon", {}).get("kills"),
            "team_towers": objectives.get("tower", {}).get("kills"),
            "surrendered": p.get("gameEndedInSurrender"),
            "early_surrender": p.get("gameEndedInEarlySurrender"),
            "summoner1_id": p.get("summoner1Id"), "summoner2_id": p.get("summoner2Id"),
            "items": [p.get(f"item{i}") for i in range(7)],
        })

    # --- 2 & 3. Minute frames + events (from timeline) ---
    info = t_data.get("info", {})
    participant_map = {p.get("participantId"): p.get("puuid") for p in info.get("participants", [])}

    for minute, frame in enumerate(info.get("frames", [])):
        for p_id_str, pf in frame.get("participantFrames", {}).items():
            p_id = int(p_id_str)
            stats = pf.get("championStats", {})
            dmg = pf.get("damageStats", {})
            pos = pf.get("position", {})
            frames_rows.append({
                "match_id": match_id,
                "minute": minute,
                "timestamp_ms": frame.get("timestamp"),
                "participant_id": p_id,
                "is_target_player": (participant_map.get(p_id) == puuid),
                "current_gold": pf.get("currentGold"),
                "total_gold": pf.get("totalGold"),
                "gold_per_sec": pf.get("goldPerSecond"),
                "level": pf.get("level"),
                "xp": pf.get("xp"),
                "lane_cs": pf.get("minionsKilled"),
                "jungle_cs": pf.get("jungleMinionsKilled"),
                "total_cs": (pf.get("minionsKilled", 0) + pf.get("jungleMinionsKilled", 0)),
                "pos_x": pos.get("x"),
                "pos_y": pos.get("y"),
                "health": stats.get("health"),
                "max_health": stats.get("healthMax"),
                "mana_energy": stats.get("power"),
                "max_mana_energy": stats.get("powerMax"),
                "ad": stats.get("attackDamage"),
                "ap": stats.get("abilityPower"),
                "armor": stats.get("armor"),
                "mr": stats.get("magicResist"),
                "attack_speed": stats.get("attackSpeed"),
                "ability_haste": stats.get("abilityHaste"),
                "movement_speed": stats.get("movementSpeed"),
                "armor_pen": stats.get("armorPen"),
                "armor_pen_pct": stats.get("armorPenPercent"),
                "magic_pen": stats.get("magicPen"),
                "magic_pen_pct": stats.get("magicPenPercent"),
                "omnivamp": stats.get("omnivamp"),
                "lifesteal": stats.get("lifesteal"),
                "total_dmg_to_champs": dmg.get("totalDamageDoneToChampions"),
                "physical_dmg_to_champs": dmg.get("physicalDamageDoneToChampions"),
                "magic_dmg_to_champs": dmg.get("magicDamageDoneToChampions"),
                "true_dmg_to_champs": dmg.get("trueDamageDoneToChampions"),
                "total_dmg_taken": dmg.get("totalDamageTaken"),
                "cc_duration_inflicted": pf.get("timeEnemySpentControlled"),
            })

        for ev in frame.get("events", []):
            pos = ev.get("position", {})
            events_rows.append({
                "match_id": match_id,
                "minute": minute,
                "timestamp_ms": ev.get("timestamp"),
                "event_type": ev.get("type"),
                "participant_id": ev.get("participantId"),
                "killer_id": ev.get("killerId"),
                "victim_id": ev.get("victimId"),
                "assisting_ids": str(ev.get("assistingParticipantIds", [])),
                "killer_team_id": ev.get("killerTeamId"),
                "item_id": ev.get("itemId"),
                "skill_slot": ev.get("skillSlot"),
                "level_up_type": ev.get("levelUpType"),
                "ward_type": ev.get("wardType"),
                "creator_id": ev.get("creatorId"),
                "monster_type": ev.get("monsterType"),
                "monster_sub_type": ev.get("monsterSubType"),
                "building_type": ev.get("buildingType"),
                "tower_type": ev.get("towerType"),
                "lane_type": ev.get("laneType"),
                "team_id": ev.get("teamId"),
                "bounty": ev.get("bounty"),
                "shutdown_bounty": ev.get("shutdownBounty"),
                "pos_x": pos.get("x"),
                "pos_y": pos.get("y"),
            })

    if idx % 10 == 0 or idx == len(riot_match_ids):
        elapsed = round(time.time() - start_time, 1)
        print(f"[{idx}/{len(riot_match_ids)}] Processed {match_id} | Elapsed: {elapsed}s")

# ===========================================================================
# 5. WRITE OUT ALL DATA FILES
# ===========================================================================
print("\nExporting datasets...")

df_players = pd.DataFrame(players_rows)
df_players.to_csv(os.path.join(DATA_DIR, "match_players_summary.csv"), index=False)
print(f"📁 1. Player summary: {len(df_players):,} rows -> match_players_summary.csv")

df_frames = pd.DataFrame(frames_rows)
df_frames.to_csv(os.path.join(DATA_DIR, "all_players_minute_frames.csv"), index=False)
print(f"📁 2. Minute frames: {len(df_frames):,} rows -> all_players_minute_frames.csv")

df_events = pd.DataFrame(events_rows)
df_events.to_csv(os.path.join(DATA_DIR, "all_matches_events_log.csv"), index=False)
print(f"📁 3. Events log: {len(df_events):,} rows -> all_matches_events_log.csv")

with gzip.open(os.path.join(DATA_DIR, "complete_raw_match_dumps.json.gz"), "wt", encoding="utf-8") as f:
    json.dump(raw_archive, f)
print("📦 4. Raw archive (match details + timelines) -> complete_raw_match_dumps.json.gz")

## Step 2 (optional): pull your OP.GG history

Adds rank over time, OP.GG's performance score and session patterns to the ranked review. Uses the unofficial [opgg.py](https://pypi.org/project/opgg.py/) library. Tip: press **Update** on your OP.GG profile first so its rank data is current.

In [ ]:
%pip install -q opgg.py==3.1.0 nest_asyncio

import nest_asyncio
nest_asyncio.apply()

import json
import os
import pandas as pd
import opgg.opgg
from opgg.opgg import OPGG
from opgg.params import Region, GameType

REGION_ENUM = getattr(Region, REGION)
GAME_TYPE = GameType.TOTAL          # GameType.TOTAL (everything), GameType.RANKED (solo + flex), GameType.NORMAL
LOCAL_TIMEZONE = TIMEZONE

# opgg.py caps requests at 200 games by default; raise the cap to whatever you asked for.
opgg.opgg._MAX_TOTAL_GAMES = max(getattr(opgg.opgg, "_MAX_TOTAL_GAMES", 200), MATCHES_TO_PULL)

opgg_client = OPGG()


def as_dict(obj):
    """Turn a pydantic model / plain object / dict into a dict."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return getattr(obj, "__dict__", {})


# 1. Build Champion ID -> Name Map
print("1. Loading champion catalog...")
champ_map = {}
try:
    for c in opgg_client.get_all_champions():
        c_dict = as_dict(c)
        cid = c_dict.get("id") or getattr(c, "id", None)
        cname = c_dict.get("name") or getattr(c, "name", None)
        if cid and cname:
            champ_map[cid] = cname
except Exception as e:
    print(f"   ⚠️ Champion catalog failed ({e}); champions will show as IDs.")

# 2. Search for Account (pick the exact Riot ID, not just the first hit)
riot_id = f"{GAME_NAME}#{TAG_LINE}"
print(f"2. Locating {riot_id}...")
search_results = opgg_client.search(riot_id, REGION_ENUM)
if not search_results:
    raise Exception(f"No OP.GG profile found for {riot_id} in {REGION}")


def summoner_of(result):
    return as_dict(getattr(result, "summoner", None) or as_dict(result).get("summoner"))


target_profile = next(
    (r for r in search_results
     if str(summoner_of(r).get("game_name", "")).lower() == GAME_NAME.lower()
     and str(summoner_of(r).get("tagline", "")).lower() == TAG_LINE.lower()),
    search_results[0],
)
target_summoner = summoner_of(target_profile)
TARGET_SUMMONER_ID = target_summoner.get("summoner_id")
TARGET_ID = target_summoner.get("id")
print(f"   ✅ Found {target_summoner.get('game_name')}#{target_summoner.get('tagline')} (summoner_id {str(TARGET_SUMMONER_ID)[:12]}...)")

# 3. Pull matches
print(f"3. Pulling up to {MATCHES_TO_PULL} matches ({GAME_TYPE.name}) from OP.GG...")
all_games = opgg_client.get_recent_games(
    search_result=target_profile,
    results=MATCHES_TO_PULL,
    game_type=GAME_TYPE,
)
print(f"🎉 Pulled {len(all_games)} matches!\n")

# 4. Parse Every Match
parsed_matches = []
raw_json_dump = []

for game in all_games:
    g = as_dict(game)
    raw_json_dump.append(g)

    game_id = g.get("id")
    created = pd.to_datetime(str(g.get("created_at", "")), errors="coerce", utc=True)
    created_at = created.tz_convert(LOCAL_TIMEZONE).strftime("%Y-%m-%d") if pd.notna(created) else ""
    game_len = g.get("game_length_second", 0) or 0
    duration_str = f"{game_len // 60}m {game_len % 60}s"
    queue = g.get("game_type", "SOLORANKED")
    is_remake = g.get("is_remake", False)

    # Find user in participants
    user_p = None
    for p in g.get("participants", []):
        p_dict = as_dict(p)
        s_dict = as_dict(p_dict.get("summoner", {}))
        if ((TARGET_SUMMONER_ID and s_dict.get("summoner_id") == TARGET_SUMMONER_ID)
                or (TARGET_ID and s_dict.get("id") == TARGET_ID)
                or (str(s_dict.get("game_name", "")).lower() == GAME_NAME.lower()
                    and str(s_dict.get("tagline", "")).lower() == TAG_LINE.lower())):
            user_p = p_dict
            break
    if not user_p:
        continue

    champ_id = user_p.get("champion_id")
    champ_name = champ_map.get(champ_id, f"ID #{champ_id}")
    role = user_p.get("position") or user_p.get("role") or "N/A"

    stats_dict = as_dict(user_p.get("stats", {}))
    k = stats_dict.get("kill") if stats_dict.get("kill") is not None else user_p.get("kill", 0)
    d = stats_dict.get("death") if stats_dict.get("death") is not None else user_p.get("death", 0)
    a = stats_dict.get("assist") if stats_dict.get("assist") is not None else user_p.get("assist", 0)
    kda_ratio = round((k + a) / max(1, d), 2)

    cs = stats_dict.get("cs", 0)
    if not cs:
        cs = (stats_dict.get("minion_kill", 0) or 0) + (stats_dict.get("neutral_minion_kill", 0) or 0)
    cs_min = round(cs / (game_len / 60), 1) if game_len > 0 else 0

    dmg = stats_dict.get("damage_dealt_to_champions") or stats_dict.get("total_damage_dealt_to_champions") or stats_dict.get("damage", 0)
    gold = stats_dict.get("gold_earned", 0)
    vision_score = stats_dict.get("vision_score", 0)
    wards_placed = stats_dict.get("ward_place", stats_dict.get("ward_placed", 0))  # OP.GG's field is "ward_place"

    if is_remake or game_len < 300:
        outcome = "REMAKE"
    else:
        is_win = stats_dict.get("is_win")
        result_str = stats_dict.get("result")
        if is_win is True or result_str == "WIN":
            outcome = "WIN"
        elif is_win is False or result_str in ("LOSE", "LOSS"):
            outcome = "LOSS"
        else:
            team_win = None
            for team in g.get("teams", []):
                t_dict = as_dict(team)
                if t_dict.get("key") == user_p.get("team_key"):
                    team_win = as_dict(t_dict.get("game_stat", {})).get("is_win")
                    break
            outcome = "WIN" if team_win else ("LOSS" if team_win is False else "REMAKE")

    parsed_matches.append({
        "Date": created_at,
        "Queue": queue,
        "Result": outcome,
        "Champion": champ_name,
        "Role": role,
        "Kills": k,
        "Deaths": d,
        "Assists": a,
        "KDA_Ratio": kda_ratio,
        "CS": cs,
        "CS_per_Min": cs_min,
        "Damage": dmg,
        "Gold": gold,
        "Vision_Score": vision_score,
        "Wards_Placed": wards_placed,
        "Duration": duration_str,
        "Game_ID": game_id,
    })

df = pd.DataFrame(parsed_matches)

# -------------------------------------------------------------
# 5. EXPORT FILES TO COLAB DISK
# -------------------------------------------------------------
csv_filename = os.path.join(DATA_DIR, f"{GAME_NAME}_season_matches.csv")
df.to_csv(csv_filename, index=False)
print(f"📁 Saved personal match history spreadsheet to: {csv_filename}")

json_filename = os.path.join(DATA_DIR, "all_matches_full_details.json")
with open(json_filename, "w") as f:
    json.dump(raw_json_dump, f, default=str, indent=2)
print(f"📦 Saved raw 10-player data for all {len(all_games)} matches to: {json_filename}\n")

# -------------------------------------------------------------
# 6. PERFORMANCE SUMMARY
# -------------------------------------------------------------
valid_games = df[df["Result"].isin(["WIN", "LOSS"])]
overall_wins = (valid_games["Result"] == "WIN").sum()
overall_games = len(valid_games)
overall_wr = round((overall_wins / overall_games) * 100, 1) if overall_games > 0 else 0

print("=" * 80)
print(f" SUMMARY FOR {riot_id} ({overall_games} games, {GAME_TYPE.name.lower()}, remakes excluded)")
print(f" Overall Win Rate: {overall_wr}% ({overall_wins}W - {overall_games - overall_wins}L)")
print("=" * 80)

if overall_games:
    champ_summary = valid_games.groupby("Champion").agg(
        Games=("Result", "count"),
        Wins=("Result", lambda x: (x == "WIN").sum()),
        WinRate=("Result", lambda x: f"{round((x == 'WIN').mean() * 100, 1)}%"),
        Avg_Kills=("Kills", "mean"),
        Avg_Deaths=("Deaths", "mean"),
        Avg_Assists=("Assists", "mean"),
        Avg_CS=("CS", "mean"),
        Avg_Dmg=("Damage", "mean"),
    ).sort_values(by="Games", ascending=False)
    for col in ["Avg_Kills", "Avg_Deaths", "Avg_Assists"]:
        champ_summary[col] = champ_summary[col].round(1)
    champ_summary["Avg_CS"] = champ_summary["Avg_CS"].round(0).astype(int)
    champ_summary["Avg_Dmg"] = champ_summary["Avg_Dmg"].round(0).astype(int)

    print("\n--- TOP CHAMPIONS ---")
    print(champ_summary.head(10).to_string())

    print("\n--- RECENT 5 MATCHES IN SPREADSHEET ---")
    print(df[["Date", "Result", "Champion", "Kills", "Deaths", "Assists", "CS", "Damage"]].head(5).to_string(index=False))

## Step 3: dashboard code

These cells only write files (`lol_process.py`, `lol_render.py` and the two page templates). Nothing to change here.

In [ ]:
%%writefile lol_process.py
"""Turn Riot API + OP.GG exports into the tables the two dashboards read.

Riot export (from the Riot pull cell):  match_players_summary.csv, all_players_minute_frames.csv,
                                        all_matches_events_log.csv  (+ optional complete_raw_match_dumps.json.gz)
OP.GG export (from the OP.GG pull cell): all_matches_full_details.json  (+ optional <name>_season_matches.csv)
"""
import ast
import gzip
import json
import os

import numpy as np
import pandas as pd

POS_NAME = {"TOP": "Top", "UTILITY": "Support", "BOTTOM": "ADC", "JUNGLE": "Jungle", "MIDDLE": "Mid"}
PRETTY = {"AurelionSol": "Aurelion Sol", "Kaisa": "Kai'Sa", "MonkeyKing": "Wukong", "Chogath": "Cho'Gath", "KSante": "K'Sante",
          "Velkoz": "Vel'Koz", "Khazix": "Kha'Zix", "Belveth": "Bel'Veth", "RekSai": "Rek'Sai", "KogMaw": "Kog'Maw",
          "TahmKench": "Tahm Kench", "MissFortune": "Miss Fortune", "XinZhao": "Xin Zhao", "JarvanIV": "Jarvan IV",
          "DrMundo": "Dr. Mundo", "LeeSin": "Lee Sin", "MasterYi": "Master Yi", "TwistedFate": "Twisted Fate",
          "Nunu": "Nunu & Willump", "Renata": "Renata Glasc", "Leblanc": "LeBlanc", "FiddleSticks": "Fiddlesticks"}
pretty = lambda c: PRETTY.get(c, c)


# =============================================================================
# Riot API export
# =============================================================================
def _zone(x, y, team):
    """Rough map area of a death, from Summoner's Rift coordinates."""
    if pd.isna(x):
        return "Unknown"
    s = x + y
    own = s < 14700 if team == 100 else s > 15000
    top = (x < 2500 and y > 4000) or (y > 12300 and x < 11000)
    bot = (y < 2500 and x > 4000) or (x > 12300 and y < 11000)
    midl = abs(x - y) < 1500 and 3500 < s < 26000
    base = (x < 4000 and y < 4000) or (x > 10800 and y > 10800)
    if base: return "Base"
    if top: return "Top lane"
    if bot: return "Bot lane"
    if midl: return "Mid lane"
    if abs(s - 14800) < 1600: return "River"
    return "Own jungle" if own else "Enemy jungle"


def load_riot(data_dir):
    paths = {k: os.path.join(data_dir, f) for k, f in [("players", "match_players_summary.csv"),
             ("frames", "all_players_minute_frames.csv"), ("events", "all_matches_events_log.csv")]}
    if not all(os.path.exists(p) for p in paths.values()):
        return None
    p = pd.read_csv(paths["players"]); f = pd.read_csv(paths["frames"]); e = pd.read_csv(paths["events"])
    p["game_start_utc"] = pd.to_datetime(p.game_start_utc, utc=True, format="ISO8601")
    p["is_target_player"] = p.is_target_player.astype(str).str.lower() == "true"
    p["win"] = p.win.astype(str).str.lower() == "true"
    p["early_surrender"] = p.early_surrender.astype(str).str.lower() == "true"
    return p, f, e


def process_riot(data_dir, tz="America/New_York"):
    """Returns a dict of DataFrames, or None when the Riot export files aren't there."""
    loaded = load_riot(data_dir)
    if loaded is None:
        return None
    p, f, e = loaded
    out = {}

    me = p[p.is_target_player].copy()
    me = me[(me.game_duration_s >= 300) & (~me.early_surrender)]
    me = me.drop_duplicates("match_id")
    games = set(me.match_id)
    me["role"] = me.position.map(POS_NAME)

    opp = p.merge(me[["match_id", "position", "team_id"]], on=["match_id", "position"], suffixes=("", "_me"))
    opp = opp[opp.team_id != opp.team_id_me][["match_id", "participant_id", "champion"]].rename(columns={"participant_id": "opp_pid", "champion": "opp_champ"})
    me = me.merge(opp.drop_duplicates("match_id"), on="match_id", how="left")
    team_of = p.set_index(["match_id", "participant_id"]).team_id

    fr = f[f.match_id.isin(games)].copy()

    def at(df, pid_col, minute, cols):
        x = fr[fr.minute == minute][["match_id", "participant_id"] + cols]
        return df.merge(x.rename(columns={"participant_id": pid_col, **{c: f"{c}_{pid_col}_{minute}" for c in cols}}), on=["match_id", pid_col], how="left")

    me = me.rename(columns={"participant_id": "pid"})
    for mnt in (10, 15):
        me = at(me, "pid", mnt, ["total_gold", "total_cs", "xp", "level"])
        me = at(me, "opp_pid", mnt, ["total_gold", "total_cs", "xp", "level"])
        me[f"gd{mnt}"] = me[f"total_gold_pid_{mnt}"] - me[f"total_gold_opp_pid_{mnt}"]
        me[f"csd{mnt}"] = me[f"total_cs_pid_{mnt}"] - me[f"total_cs_opp_pid_{mnt}"]
        me[f"cs{mnt}"] = me[f"total_cs_pid_{mnt}"]

    # team gold difference per minute
    fr["team"] = [team_of.get((m, pid)) for m, pid in zip(fr.match_id, fr.participant_id)]
    tg = fr.groupby(["match_id", "minute", "team"]).total_gold.sum().unstack("team")
    me_team = me.set_index("match_id").team_id
    tg["tgd"] = [row.get(me_team[m], np.nan) - row.get(300 - me_team[m], np.nan) for (m, _), row in tg.iterrows()]
    tgd = tg.tgd.unstack("minute")
    for mnt in (10, 15):
        me[f"tgd{mnt}"] = me.match_id.map(tgd[mnt]) if mnt in tgd else np.nan

    # your deaths
    k = e[(e.event_type == "CHAMPION_KILL") & e.match_id.isin(games)].copy()
    k["assists_n"] = k.assisting_ids.apply(lambda s: len(ast.literal_eval(s)) if isinstance(s, str) else 0)
    mp = me[["match_id", "pid", "team_id", "win", "role", "champion"]].rename(columns={"team_id": "my_team"})
    d = k.merge(mp, left_on=["match_id", "victim_id"], right_on=["match_id", "pid"])
    d["t"] = d.timestamp_ms / 60000
    d["phase"] = pd.cut(d.t, [0, 14, 25, 1000], labels=["0–14", "14–25", "25+"])
    d["attackers"] = 1 + d.assists_n
    d["zone"] = [_zone(x, y, t) for x, y, t in zip(d.pos_x, d.pos_y, d.my_team)]

    obj = e[(e.event_type.isin(["ELITE_MONSTER_KILL", "BUILDING_KILL"])) & e.match_id.isin(games)].copy()
    obj = obj.merge(me[["match_id", "team_id"]].rename(columns={"team_id": "my_team"}), on="match_id")
    # BUILDING_KILL's team_id is the team that LOST the building
    obj["obj_team"] = np.where(obj.event_type == "ELITE_MONSTER_KILL",
                               [team_of.get((m, k_)) for m, k_ in zip(obj.match_id, obj.killer_id)],
                               np.where(obj.team_id == 100, 200, 100))
    enemy_obj = obj[obj.obj_team != obj.my_team]
    eo = {m: g.timestamp_ms.values for m, g in enemy_obj.groupby("match_id")}
    d["gave_obj"] = [bool(((eo.get(m, np.array([])) > t) & (eo.get(m, np.array([])) <= t + 60000)).any()) for m, t in zip(d.match_id, d.timestamp_ms)]

    # fights your team won (2+ kills, <20s apart) and whether an objective followed within 90s
    mk = k.merge(me[["match_id", "team_id"]].rename(columns={"team_id": "my_team"}), on="match_id")
    mk["kteam"] = [team_of.get((m, kid)) for m, kid in zip(mk.match_id, mk.killer_id)]
    mine = mk[mk.kteam == mk.my_team].sort_values(["match_id", "timestamp_ms"])
    my_obj = obj[obj.obj_team == obj.my_team]
    mo = {m: g.timestamp_ms.values for m, g in my_obj.groupby("match_id")}
    fights = []
    for mid, g in mine.groupby("match_id"):
        ts = g.timestamp_ms.values; i = 0
        while i < len(ts):
            j = i
            while j + 1 < len(ts) and ts[j + 1] - ts[j] <= 20000:
                j += 1
            if j - i + 1 >= 2:
                o = mo.get(mid, np.array([]))
                fights.append((mid, j - i + 1, bool(((o > ts[j] - 20000) & (o <= ts[j] + 90000)).any())))
            i = j + 1
    fights = pd.DataFrame(fights, columns=["match_id", "kills", "converted"])
    fights["win"] = fights.match_id.map(me.set_index("match_id").win)

    bk = obj[obj.event_type == "BUILDING_KILL"]
    ft = bk.sort_values("timestamp_ms").groupby("match_id").first()
    me["first_tower_mine"] = me.match_id.map(ft.obj_team) == me.team_id
    me["date"] = me.game_start_utc.dt.tz_convert(tz).dt.strftime("%Y-%m-%d")

    # ---- timeline_games / deaths / minutes / fights ----
    fd = d.sort_values("t").groupby("match_id").t.first()
    ph = d.groupby(["match_id", "phase"], observed=False).size().unstack("phase").reindex(me.match_id).fillna(0).astype(int)
    for col in ["0–14", "14–25", "25+"]:
        if col not in ph: ph[col] = 0
    out["timeline_games"] = pd.DataFrame({"match_id": me.match_id, "date": me.date, "role": me.role, "champion": me.champion.map(pretty),
        "win": me.win.astype(int), "minutes": (me.game_duration_s / 60).round(1),
        "lane_gold_10": me.gd10.round(0), "lane_cs_10": me.csd10, "lane_gold_15": me.gd15.round(0), "lane_cs_15": me.csd15, "cs_at_10": me.cs10,
        "team_gold_15": me.tgd15.round(0), "first_death_min": me.match_id.map(fd).round(1),
        "deaths_0_14": ph["0–14"].values, "deaths_14_25": ph["14–25"].values, "deaths_25_plus": ph["25+"].values,
        "first_tower_ours": me.first_tower_mine.astype(int)}).sort_values("date")
    out["timeline_deaths"] = pd.DataFrame({"match_id": d.match_id, "minute": d.t.round(2), "role": d.role, "zone": d.zone, "attackers": d.attackers,
                                           "win": d.win.astype(int), "enemy_objective_within_60s": d.gave_obj.astype(int)})
    pid = me.set_index("match_id")[["pid", "opp_pid", "role", "win"]]
    x = fr[["match_id", "minute", "participant_id", "total_gold"]]
    gmine = x.merge(pid.reset_index()[["match_id", "pid"]], left_on=["match_id", "participant_id"], right_on=["match_id", "pid"])[["match_id", "minute", "total_gold"]]
    gopp = x.merge(pid.reset_index()[["match_id", "opp_pid"]], left_on=["match_id", "participant_id"], right_on=["match_id", "opp_pid"])[["match_id", "minute", "total_gold"]]
    lane = gmine.merge(gopp, on=["match_id", "minute"], suffixes=("_me", "_opp")); lane["lane_gold_diff"] = lane.total_gold_me - lane.total_gold_opp
    c = tgd.stack().rename("team_gold_diff").reset_index().merge(lane[["match_id", "minute", "lane_gold_diff"]], on=["match_id", "minute"], how="left")
    c = c[c.minute <= 35]
    c["win"] = c.match_id.map(pid.win).astype(int); c["role"] = c.match_id.map(pid.role)
    out["timeline_minutes"] = c[["match_id", "minute", "role", "win", "team_gold_diff", "lane_gold_diff"]]
    out["timeline_fights"] = fights.assign(converted=fights.converted.astype(int), win=fights.win.astype(int))[["match_id", "kills", "converted", "win"]]

    # ---- matchups / champ_games ----
    gold = fr.set_index(["match_id", "minute", "participant_id"]).total_gold

    def g_at(m, mn, pids):
        try:
            return float(sum(gold.loc[(m, mn, int(pp))] for pp in pids))
        except KeyError:
            return np.nan

    team_tot = p.groupby(["match_id", "team_id"]).agg(tk=("kills", "sum"), td=("dmg_to_champs", "sum"))
    early = d[d.t < 14].groupby("match_id").size()
    rows = []
    for r in me.itertuples():
        m = r.match_id; gp = p[p.match_id == m]
        ally = gp[gp.team_id == r.team_id]; en = gp[gp.team_id != r.team_id]

        def one(df, ps):
            q = df[df.position == ps]
            return q.iloc[0] if len(q) else None
        o = one(en, r.position)
        tk, td = team_tot.loc[(m, r.team_id)]
        mins = r.game_duration_s / 60
        row = dict(match_id=m, date=r.date, role=r.role, champion=pretty(r.champion), win=int(r.win), minutes=round(mins, 1),
                   kills=r.kills, deaths=r.deaths, assists=r.assists, cs_per_min=round(r.total_cs / mins, 2),
                   kill_participation=round((r.kills + r.assists) / tk, 3) if tk else np.nan,
                   damage_share=round(r.dmg_to_champs / td, 3) if td else np.nan, vision_per_min=round(r.vision_score / mins, 2),
                   opponent=pretty(o.champion) if o is not None else "", enemy_adc="", enemy_support="", partner="",
                   team_gold_15=r.tgd15, early_deaths=int(early.get(m, 0)))
        for mn in (10, 15):
            row[f"lane_gold_{mn}"] = g_at(m, mn, [r.pid]) - g_at(m, mn, [o.participant_id]) if o is not None else np.nan
            row[f"duo_gold_{mn}"] = np.nan
        if r.position in ("BOTTOM", "UTILITY"):
            ea, es = one(en, "BOTTOM"), one(en, "UTILITY"); partner = one(ally, "UTILITY" if r.position == "BOTTOM" else "BOTTOM")
            row.update(enemy_adc=pretty(ea.champion) if ea is not None else "", enemy_support=pretty(es.champion) if es is not None else "",
                       partner=pretty(partner.champion) if partner is not None else "")
            if ea is not None and es is not None and partner is not None:
                for mn in (10, 15):
                    row[f"duo_gold_{mn}"] = g_at(m, mn, [r.pid, partner.participant_id]) - g_at(m, mn, [ea.participant_id, es.participant_id])
        rows.append(row)
    cg = pd.DataFrame(rows).sort_values("date")
    for col in ["lane_gold_10", "lane_gold_15", "duo_gold_10", "duo_gold_15", "team_gold_15"]:
        cg[col] = cg[col].round().astype("Int64")
    out["champ_games"] = cg[["match_id", "date", "role", "champion", "win", "minutes", "kills", "deaths", "assists", "cs_per_min", "kill_participation",
                             "damage_share", "vision_per_min", "opponent", "enemy_adc", "enemy_support", "partner", "lane_gold_10", "lane_gold_15",
                             "duo_gold_10", "duo_gold_15", "team_gold_15", "early_deaths"]]

    # ---- per-game gold lines and event markers ----
    champ = {(q.match_id, q.participant_id): pretty(q.champion) for q in p.itertuples()}
    dname = {"EARTH_DRAGON": "Mountain dragon", "HEXTECH_DRAGON": "Hextech dragon", "WATER_DRAGON": "Ocean dragon", "CHEMTECH_DRAGON": "Chemtech dragon",
             "FIRE_DRAGON": "Infernal dragon", "AIR_DRAGON": "Cloud dragon", "ELDER_DRAGON": "Elder dragon"}
    lanen = {"TOP_LANE": "Top", "MID_LANE": "Mid", "BOT_LANE": "Bot"}
    tn = {"OUTER_TURRET": "outer", "INNER_TURRET": "inner", "BASE_TURRET": "base", "NEXUS_TURRET": "nexus"}
    grow, ev = [], []
    ev_by = {m: g for m, g in e[e.match_id.isin(games)].groupby("match_id")}
    for r in me.itertuples():
        m = r.match_id; my = r.team_id; gp = p[p.match_id == m]
        if r.position in ("BOTTOM", "UTILITY"):
            mine_ids = gp[(gp.team_id == my) & gp.position.isin(["BOTTOM", "UTILITY"])].participant_id.tolist()
            their_ids = gp[(gp.team_id != my) & gp.position.isin(["BOTTOM", "UTILITY"])].participant_id.tolist()
        else:
            mine_ids = [r.pid]; their_ids = [r.opp_pid] if not pd.isna(r.opp_pid) else []
        allied = gp[gp.team_id == my].participant_id.tolist(); enemy = gp[gp.team_id != my].participant_id.tolist()
        for mn in sorted(fr[fr.match_id == m].minute.unique()):
            try:
                gd = (sum(gold.loc[(m, mn, int(q))] for q in mine_ids) - sum(gold.loc[(m, mn, int(q))] for q in their_ids)) if their_ids else np.nan
                tgv = sum(gold.loc[(m, mn, int(q))] for q in allied) - sum(gold.loc[(m, mn, int(q))] for q in enemy)
            except KeyError:
                continue
            grow.append((m, int(mn), gd, tgv))
        ge = ev_by.get(m, e.iloc[0:0])
        for kk in ge[ge.event_type == "CHAMPION_KILL"].itertuples():
            tt = round(kk.timestamp_ms / 60000, 2)
            assists = ast.literal_eval(kk.assisting_ids) if isinstance(kk.assisting_ids, str) else []
            killer = champ.get((m, int(kk.killer_id))) if not pd.isna(kk.killer_id) and kk.killer_id > 0 else "Minion/tower"
            victim = champ.get((m, int(kk.victim_id)), "?") if not pd.isna(kk.victim_id) else "?"
            if kk.victim_id == r.pid: ev.append((m, tt, "Death", "You", "Killed by " + str(killer) + (f" + {len(assists)}" if assists else "")))
            elif kk.killer_id == r.pid: ev.append((m, tt, "Kill", "You", "Killed " + victim))
            elif r.pid in assists: ev.append((m, tt, "Assist", "You", "Helped kill " + victim))
        for o in ge[ge.event_type == "ELITE_MONSTER_KILL"].itertuples():
            tteam = o.killer_team_id if not pd.isna(o.killer_team_id) else team_of.get((m, int(o.killer_id)))
            mt = o.monster_type
            kind = {"DRAGON": "Elder dragon" if o.monster_sub_type == "ELDER_DRAGON" else "Dragon", "BARON_NASHOR": "Baron",
                    "RIFTHERALD": "Herald", "HORDE": "Grubs"}.get(mt, mt)
            det = {"DRAGON": dname.get(o.monster_sub_type, "Dragon"), "BARON_NASHOR": "Baron Nashor", "RIFTHERALD": "Rift Herald", "HORDE": "Void grub"}.get(mt, mt)
            ev.append((m, round(o.timestamp_ms / 60000, 2), kind, "Your team" if tteam == my else "Enemy team", det))
        for b in ge[ge.event_type == "BUILDING_KILL"].itertuples():
            taker = "Enemy team" if b.team_id == my else "Your team"
            if b.building_type == "INHIBITOR_BUILDING": kind, det = "Inhibitor", lanen.get(b.lane_type, "") + " inhibitor"
            else: kind, det = "Tower", (lanen.get(b.lane_type, "") + " " + tn.get(b.tower_type, "") + " tower").strip()
            ev.append((m, round(b.timestamp_ms / 60000, 2), kind, taker, det))
    gg = pd.DataFrame(grow, columns=["match_id", "minute", "gold_diff", "team_gold_diff"])
    gg["gold_diff"] = gg.gold_diff.round().astype("Int64"); gg["team_gold_diff"] = gg.team_gold_diff.round().astype("Int64")
    out["game_gold"] = gg
    v = pd.DataFrame(ev, columns=["match_id", "minute", "kind", "side", "detail"]).sort_values(["match_id", "minute"])
    v["grp"] = ((v.kind != "Grubs") | (v.kind != v.kind.shift()) | (v.match_id != v.match_id.shift()) | (v.minute - v.minute.shift() > 0.5)).cumsum()
    agg = v.groupby("grp").agg(match_id=("match_id", "first"), minute=("minute", "first"), kind=("kind", "first"),
                               side=("side", "first"), detail=("detail", "first"), n=("kind", "size"))
    gm = agg.kind == "Grubs"
    agg.loc[gm, "detail"] = agg[gm].n.astype(str) + " void grub" + np.where(agg[gm].n > 1, "s", "")
    out["game_events"] = agg.drop(columns="n").reset_index(drop=True)

    # ---- objectives: were you near, and who died just before ----
    pos = f.set_index(["match_id", "minute", "participant_id"])[["pos_x", "pos_y"]]
    DRAG, TOPPIT = (9866, 4414), (5007, 10471)
    ob = e[(e.event_type == "ELITE_MONSTER_KILL") & e.match_id.isin(games)].copy()
    ob["t"] = ob.timestamp_ms / 60000
    ob["kind"] = ob.monster_type.map({"DRAGON": "Dragon", "BARON_NASHOR": "Baron", "RIFTHERALD": "Herald", "HORDE": "Grubs"})
    ob.loc[ob.monster_sub_type == "ELDER_DRAGON", "kind"] = "Elder"
    ob["taker"] = [kt if not pd.isna(kt) else team_of.get((m, int(kid))) for m, kt, kid in zip(ob.match_id, ob.killer_team_id, ob.killer_id)]
    ob = ob.sort_values(["match_id", "t"])
    ob["grp"] = ((ob.kind != "Grubs") | (ob.kind != ob.kind.shift()) | (ob.match_id != ob.match_id.shift()) | (ob.t - ob.t.shift() > 0.75)).cumsum()
    ob = ob.groupby("grp").agg(match_id=("match_id", "first"), t=("t", "first"), kind=("kind", "first"), taker=("taker", "first")).reset_index(drop=True)
    info = me.set_index("match_id")[["pid", "team_id", "win", "role"]]
    ob = ob.join(info, on="match_id")
    kills = k.copy(); kills["t"] = kills.timestamp_ms / 60000
    kills["vteam"] = [team_of.get((m, int(vv))) if not pd.isna(vv) and vv > 0 else None for m, vv in zip(kills.match_id, kills.victim_id)]
    kg = {m: g for m, g in kills.groupby("match_id")}
    orow = []
    for o in ob.itertuples():
        pit = DRAG if o.kind in ("Dragon", "Elder") else TOPPIT
        try:
            px, py = pos.loc[(o.match_id, int(np.floor(o.t)), o.pid)]
        except KeyError:
            px = py = np.nan
        dist = float(np.hypot(px - pit[0], py - pit[1]))
        w = kg.get(o.match_id, kills.iloc[0:0]); w = w[(w.t > o.t - 1.5) & (w.t <= o.t)]
        orow.append(dict(match_id=o.match_id, minute=round(o.t, 2), objective=o.kind, your_team_took_it=int(o.taker == o.team_id), game_won=int(o.win),
                         role=o.role, your_distance=round(dist) if not np.isnan(dist) else np.nan, you_near=int(dist < 4000),
                         you_died_90s_before=int((w.victim_id == o.pid).any()), team_deaths_90s_before=int((w.vteam == o.team_id).sum()),
                         enemy_deaths_90s_before=int(((w.vteam.notna()) & (w.vteam != o.team_id)).sum())))
    out["objectives"] = pd.DataFrame(orow)
    return out


# =============================================================================
# OP.GG export
# =============================================================================
def _champion_names(data_dir, game_name):
    """Champion id -> name from the OP.GG CSV (by game) and the Riot raw dump, when present."""
    names_by_game, by_id = {}, {}
    csv = os.path.join(data_dir, f"{game_name}_season_matches.csv")
    if not os.path.exists(csv):
        csv = next((os.path.join(data_dir, x) for x in sorted(os.listdir(data_dir)) if x.endswith("_season_matches.csv")), None)
    if csv and os.path.exists(csv):
        c = pd.read_csv(csv)
        names_by_game = dict(zip(c.Game_ID, c.Champion))
    raw = os.path.join(data_dir, "complete_raw_match_dumps.json.gz")
    if os.path.exists(raw):
        try:
            for g in json.load(gzip.open(raw, "rt")):
                m = g.get("match", g)
                for q in m.get("info", {}).get("participants", []):
                    by_id[q["championId"]] = pretty(q["championName"])
        except Exception:
            pass
    return names_by_game, by_id


def process_opgg(data_dir, game_name, tag_line="", tz="America/New_York"):
    """Returns the ranked_games DataFrame, or None when the OP.GG JSON isn't there."""
    path = os.path.join(data_dir, "all_matches_full_details.json")
    if not os.path.exists(path):
        return None
    data = json.load(open(path))
    names_by_game, by_id = _champion_names(data_dir, game_name)
    gn, tl = game_name.lower(), tag_line.lower()

    def is_me(q):
        s = q.get("summoner") or {}
        return str(s.get("game_name", "")).lower() == gn and (not tl or str(s.get("tagline", "")).lower() == tl)

    rows = []
    for m in data:
        parts = m.get("participants") or []
        me = next((q for q in parts if is_me(q)), None)
        if me is None:
            continue
        s = me["stats"]; team = me["team_key"]
        allies = [q for q in parts if q["team_key"] == team]; enemies = [q for q in parts if q["team_key"] != team]
        tstat = next(t["game_stat"] for t in m["teams"] if t["key"] == team)
        opp = next((q for q in enemies if q["position"] == me["position"] and me["position"]), None)
        mins = (m.get("game_length_second") or 0) / 60
        cs = (s.get("minion_kill") or 0) + (s.get("neutral_minion_kill") or 0)
        tline = {pt["second"]: pt["score"] for pt in (s.get("op_score_timeline") or [])}

        def at(sec):
            if not tline: return None
            keys = [kk for kk in tline if kk <= sec]
            return tline[max(keys)] if keys and sec <= max(tline) else None
        ally_dmg = sum(q["stats"]["total_damage_dealt_to_champions"] for q in allies)
        tk = tstat.get("champion_kill") or 0
        ti = me.get("tier_info") or {}
        name = names_by_game.get(m["id"]) or by_id.get(me["champion_id"]) or f"ID #{me['champion_id']}"
        rows.append(dict(
            id=m["id"], start=pd.to_datetime(str(m["created_at"]), utc=True).tz_convert(tz), queue=m.get("game_type"),
            remake=bool(m.get("is_remake")), minutes=mins, champion=name, position=me.get("position"),
            win=s.get("result") == "WIN", k=s["kill"], d=s["death"], a=s["assist"], cspm=cs / mins if mins else 0,
            gold=s["gold_earned"], vspm=s["vision_score"] / mins if mins else 0,
            kp=(s["kill"] + s["assist"]) / tk if tk else None,
            dmg_share=s["total_damage_dealt_to_champions"] / ally_dmg if ally_dmg else None,
            op_score=s.get("op_score"), op_rank=s.get("op_score_rank"),
            op5=at(300), op10=at(600), op15=at(900), op20=at(1200), op25=at(1500),
            first_blood=tstat.get("champion_first"), first_drag=tstat.get("dragon_first"), first_tower=tstat.get("tower_first"),
            first_grubs=tstat.get("horde_first"), first_herald=tstat.get("rift_herald_first"), first_baron=tstat.get("baron_first"),
            tier=ti.get("tier"), div=ti.get("division"), lp=ti.get("lp"),
            opp_gold=opp["stats"]["gold_earned"] if opp else None))
    if not rows:
        return None
    r = pd.DataFrame(rows).drop_duplicates("id").sort_values("start").reset_index(drop=True)
    r = r[(r.queue == "SOLORANKED") & (~r.remake) & (r.minutes > 5)].reset_index(drop=True)
    if r.empty:
        return None
    order = {"IRON": 0, "BRONZE": 1, "SILVER": 2, "GOLD": 3, "PLATINUM": 4, "EMERALD": 5, "DIAMOND": 6}
    r["ladder"] = [order[t] * 400 + (4 - dv) * 100 + (lp or 0) if t in order and not pd.isna(dv) else
                   (2800 + (lp or 0) if t in ("MASTER", "GRANDMASTER", "CHALLENGER") else np.nan)
                   for t, dv, lp in zip(r["tier"], r["div"], r["lp"])]
    gap = r.start.diff().dt.total_seconds().div(60)
    r["session"] = (gap.isna() | (gap > 90)).cumsum()
    r["gidx"] = r.groupby("session").cumcount() + 1
    flag = lambda col: r[col].fillna(False).astype(bool).astype(int)
    return pd.DataFrame({
        "game": np.arange(1, len(r) + 1), "date": r.start.dt.strftime("%Y-%m-%d"), "hour_et": r.start.dt.hour,
        "weekday": r.start.dt.day_name().str[:3], "session_game": r.gidx, "champion": r.champion,
        "role": r.position.fillna("").str.title().replace({"Adc": "ADC"}), "win": r.win.astype(int), "minutes": r.minutes.round(1),
        "kills": r.k, "deaths": r.d, "assists": r.a, "cs_per_min": r.cspm.round(2), "kill_participation": r.kp.astype(float).round(3),
        "damage_share": r.dmg_share.astype(float).round(3), "vision_per_min": r.vspm.round(2), "op_score": r.op_score.astype(float).round(2),
        "op_rank": r.op_rank, "op_5": r.op5.astype(float).round(2), "op_10": r.op10.astype(float).round(2), "op_15": r.op15.astype(float).round(2),
        "op_20": r.op20.astype(float).round(2), "op_25": r.op25.astype(float).round(2),
        "first_blood": flag("first_blood"), "first_dragon": flag("first_drag"), "first_grubs": flag("first_grubs"),
        "first_herald": flag("first_herald"), "first_tower": flag("first_tower"), "first_baron": flag("first_baron"),
        "lane_gold_diff": (r.gold - r.opp_gold).round(0), "rank_points": r.ladder, "tier": r["tier"].fillna("").str.title(),
        "division": r["div"].astype("Int64"), "lp": r["lp"].astype("Int64")})


def load_ratings(data_dir):
    """Optional matchup ratings (role, champion, enemy, edge, games, patch) for the ★ marks."""
    path = os.path.join(data_dir, "matchup_ratings.csv")
    return pd.read_csv(path) if os.path.exists(path) else None

In [ ]:
%%writefile lol_render.py
"""Render the dashboard templates in a notebook cell (and as standalone .html files)."""
import html as _html
import json
import re
import urllib.request

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7.9.0/dist/d3.min.js"
_D3 = None


def _d3_tag():
    """Inline d3 so the page works offline and inside notebook output frames; fall back to the CDN tag."""
    global _D3
    if _D3 is None:
        try:
            with urllib.request.urlopen(D3_URL, timeout=20) as r:
                _D3 = "<script>" + r.read().decode("utf-8").replace("</script", "<\\/script") + "</script>"
        except Exception:
            _D3 = f'<script src="{D3_URL}"></script>'
    return _D3


THEME_CSS = """
:root {
  --color-fg: #1f2328; --color-fg-muted: #656d76; --color-bg: #ffffff; --color-panel: #f6f8fa; --color-border-line: #d8dee4;
  --color-ok: #1a7f37; --color-warn: #9a6700; --color-bad: #cf222e;
  --cds-chart-grid: #e6e9ed; --cds-chart-axis: #8c959f; --cds-chart-muted: #b6bec7; --cds-chart-reference: #8c959f;
  --cds-text-secondary: #656d76; --cds-surface-popover: #ffffff; --cds-border: #d0d7de; --cds-shadow-popover: 0 6px 18px rgba(31,35,40,.15);
  --font-anthropic-sans: system-ui, -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  --font-anthropic-serif: Georgia, "Iowan Old Style", "Times New Roman", serif;
  --cds-font-size-caption: 12px; --cds-font-size-body: 14px; --cds-font-size-heading: 16px; --cds-font-size-title: 26px;
  --cds-font-weight-medium: 600; --cds-radius: 8px;
  --cds-gap-xs: 4px; --cds-gap-sm: 8px; --cds-gap-md: 12px; --cds-gap-lg: 20px; --cds-gap-xl: 32px;
  --cds-pad-xs: 4px; --cds-pad-sm: 8px; --cds-pad-md: 12px; --cds-pad-lg: 18px;
  --cds-dur-slow: .3s;
}
@media (prefers-color-scheme: dark) {
  :root {
    --color-fg: #e6edf3; --color-fg-muted: #9198a1; --color-bg: #0d1117; --color-panel: #161b22; --color-border-line: #30363d;
    --color-ok: #3fb950; --color-warn: #d29922; --color-bad: #f85149;
    --cds-chart-grid: #262c33; --cds-chart-axis: #6e7681; --cds-chart-muted: #545d68; --cds-chart-reference: #6e7681;
    --cds-text-secondary: #9198a1; --cds-surface-popover: #1c2128; --cds-border: #3d444d; --cds-shadow-popover: 0 6px 18px rgba(0,0,0,.5);
  }
}
html, body { margin: 0; background: var(--color-bg); color: var(--color-fg); font-family: var(--font-anthropic-sans); font-size: 14px; }
#dash-root { padding: var(--dash-gutter, 18px); }
.dash-skeleton:empty { display: inline-block; min-width: 2em; }
.nb-banner { font-family: var(--font-anthropic-sans); font-size: 13px; color: var(--color-fg); background: var(--color-panel);
  border: 1px solid var(--color-border-line); border-left: 4px solid var(--color-warn); border-radius: 8px; padding: 10px 14px;
  margin: 0 auto 16px; max-width: 72rem; }
"""

# A small stand-in for the hosted dashboard runtime: datasets embedded as CSV text, calculations, params, redraws.
SHIM_JS = r"""
(() => {
  const raw = JSON.parse(document.getElementById("nb-data").textContent);
  const sets = {}, calcs = {}, memo = {}, subs = [];
  for (const [id, csv] of Object.entries(raw)) sets[id] = csv == null ? null : d3.csvParse(csv);
  let params = {};
  const run = () => subs.forEach((f) => { try { f(); } catch (e) { console.error(e); } });
  window.dash = {
    calc(id, o) { calcs[id] = o; },
    loader() {},
    data(id) {
      if (id in sets) return sets[id] == null ? { status: "missing", data: [], columns: [] } : { status: "ok", data: sets[id], columns: sets[id].columns || [] };
      const c = calcs[id];
      if (!c) return { status: "missing", data: [], columns: [] };
      if (memo[id]) return memo[id];
      const ins = c.inputs.map((i) => dash.data(i));
      const bad = ins.find((x) => x.status !== "ok");
      if (bad) return (memo[id] = { status: bad.status, data: [], columns: [] });
      try { const d = c.fn(...ins.map((x) => x.data)); return (memo[id] = { status: "ok", data: d, columns: [] }); }
      catch (e) { console.error("calc " + id, e); return (memo[id] = { status: "error", data: [], columns: [], message: String(e) }); }
    },
    onData(f) { subs.push(f); try { f(); } catch (e) { console.error(e); } },
    params() { return { ...params }; },
    setParams(o) { params = { ...params, ...o }; run(); },
    resetParams() { params = {}; run(); },
    setLink() {}, refresh() {},
    colors: ["#2f6fdf", "#d9822b", "#3aa17e", "#a259d9", "#d4463b", "#8a8f2e", "#cc4fa0", "#5b8fa8"],
  };
  let t; window.addEventListener("resize", () => { clearTimeout(t); t = setTimeout(run, 150); });
  const fit = () => { try { if (window.frameElement) window.frameElement.style.height = (document.documentElement.scrollHeight + 4) + "px"; } catch (e) {} };
  window.addEventListener("load", () => { fit(); if (window.ResizeObserver) new ResizeObserver(fit).observe(document.body); else setInterval(fit, 1000); });
})();
"""


def build_page(template, datasets, player, notes=()):
    """template: HTML text; datasets: {id: DataFrame or None}. Returns a full standalone HTML document."""
    page = template.replace("{{PLAYER}}", _html.escape(player))
    payload = {k: (None if v is None else v.to_csv(index=False)) for k, v in datasets.items()}
    data_json = json.dumps(payload).replace("</", "<\\/")
    banner = "".join(f'<div class="nb-banner">{_html.escape(n)}</div>' for n in notes)
    title = _html.escape(re.sub(r"<[^>]+>", "", re.search(r"<h1[^>]*>(.*?)</h1>", page, re.S).group(1)).replace("&amp;", "&"))
    return ("<!doctype html><html><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>"
            f"<title>{title}</title><style>{THEME_CSS}</style>{_d3_tag()}"
            f"<script id='nb-data' type='application/json'>{data_json}</script><script>{SHIM_JS}</script></head>"
            f"<body><div id='dash-root'>{banner}{page}</div></body></html>")


def show(doc, height=900):
    """Display a built page inside the notebook output (an auto-sizing iframe)."""
    from IPython.display import HTML, display
    display(HTML(f'<iframe srcdoc="{_html.escape(doc, quote=True)}" style="width:100%;height:{height}px;border:0;" loading="eager"></iframe>'))


def _hide_between(template, start, end):
    i, j = template.find(start), template.find(end)
    if i < 0 or j < 0 or j < i:
        return template
    return template[:i] + '<div hidden>' + template[i:j] + '</div>' + template[j:]


def trim_review(template, has_opgg, has_riot):
    """Hide the sections whose export is missing (their scripts still find their elements, so nothing breaks)."""
    if not has_opgg:
        template = _hide_between(template, '<p class="take">Solo/duo', '<h2 id="minute">')
    if not has_riot:
        template = _hide_between(template, '<h2 id="minute">', '<div class="tip" id="tip"></div>')
    return template


def without_ratings_note(template):
    """Drop the lolalytics note from the matchups page when no ratings file was given."""
    return re.sub(r'\s*<p class="take edge-note"[\s\S]*?</p>', "", template, count=1)

In [ ]:
%%writefile review_template.html
<style>
#dash-root { --dash-gutter: var(--cds-pad-lg); }
.rv { position: relative; font-family: var(--font-anthropic-sans); max-width: 72rem; margin-inline: auto; color: var(--color-fg); }
.rv h1 { font-family: var(--font-anthropic-serif); font-weight: 400; font-size: var(--cds-font-size-title); margin: 0 0 var(--cds-gap-xs); }
.rv .take { color: var(--color-fg-muted); font-size: var(--cds-font-size-body); margin: 0 0 var(--cds-gap-lg); max-width: 52rem; line-height: 1.5; }
.rv .take b { font-weight: 600; color: var(--color-fg); }
.rv .kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 11rem), 1fr)); gap: var(--cds-gap-md); margin-bottom: var(--cds-gap-xl); }
.rv .kpi { background: var(--color-panel); border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); padding: var(--cds-pad-md) var(--cds-pad-lg); }
.rv .kpi .lab { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); }
.rv .kpi .num { font-family: var(--font-anthropic-serif); font-weight: 400; font-size: var(--cds-font-size-title); font-variant-numeric: tabular-nums; margin: var(--cds-gap-xs) 0; }
.rv .kpi .sub { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); font-variant-numeric: tabular-nums; }
.rv h2 { font-size: var(--cds-font-size-heading); font-weight: var(--cds-font-weight-medium); margin: var(--cds-gap-xl) 0 var(--cds-gap-md); }
.rv .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 22rem), 1fr)); gap: var(--cds-gap-md); }
.rv .card { background: var(--color-panel); border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); padding: var(--cds-pad-lg); position: relative; min-width: 0; }
.rv .card h3 { font-size: var(--cds-font-size-heading); font-weight: var(--cds-font-weight-medium); margin: 0 0 var(--cds-gap-xs); }
.rv .card .note { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); margin: 0 0 var(--cds-gap-md); line-height: 1.45; }
.rv .chart { min-height: 12rem; position: relative; }
.rv .chart.tall { min-height: 16rem; }
.rv svg text { font-family: var(--font-anthropic-sans); }
.rv .tip { position: absolute; pointer-events: none; background: var(--cds-surface-popover); border: 1px solid var(--cds-border); border-radius: var(--cds-radius); box-shadow: var(--cds-shadow-popover); padding: var(--cds-pad-xs) var(--cds-pad-sm); font-size: var(--cds-font-size-caption); white-space: nowrap; z-index: 5; display: none; }
.rv .tip .t { color: var(--cds-text-secondary); }
.rv .legend { display: flex; gap: var(--cds-gap-md); font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); margin-bottom: var(--cds-gap-sm); flex-wrap: wrap; }
.rv .legend i { display: inline-block; width: .6rem; height: .6rem; border-radius: 50%; margin-right: .35rem; vertical-align: middle; }
.rv table { width: 100%; border-collapse: collapse; font-size: var(--cds-font-size-caption); font-variant-numeric: tabular-nums; }
.rv th, .rv td { text-align: right; padding: var(--cds-pad-xs) var(--cds-pad-sm); border-bottom: 1px solid var(--color-border-line); }
.rv th:first-child, .rv td:first-child { text-align: left; }
.rv th { color: var(--color-fg-muted); font-weight: var(--cds-font-weight-medium); position: sticky; top: 0; background: var(--color-panel); }
.rv .tblwrap { max-height: 26rem; overflow: auto; }
.rv .good { color: var(--color-ok); } .rv .bad { color: var(--color-bad); }
.rv .fail { display: flex; align-items: center; justify-content: center; gap: .4rem; min-height: inherit; color: var(--color-fg-muted); font-size: var(--cds-font-size-caption); }
.rv .fail::before { content: ""; width: .375rem; height: .375rem; border-radius: 50%; background: var(--color-bad); }
.rv [data-dash-source-state=empty] { color: transparent; }
.rv [data-dash-source-state=refreshing] { opacity: .5; transition: opacity var(--cds-dur-slow); }
.rv .tips { font-size: var(--cds-font-size-body); line-height: 1.55; margin: 0; padding-left: 1.2rem; }
.rv .tips li { margin-bottom: var(--cds-gap-sm); }
@media (max-width: 30rem) { .rv .hide-narrow { display: none; } }
</style>

<div class="rv">
  <h1 id="top">{{PLAYER}} · Ranked review</h1>
  <p class="take">Solo/duo, <span data-source="summary" data-field="first_date" class="dash-skeleton">—</span> to <span data-source="summary" data-field="last_date" class="dash-skeleton">—</span>. You won <b data-source="summary" data-field="wr_pct" class="dash-skeleton">—</b> of <span data-source="summary" data-field="games" class="dash-skeleton">—</span> games. You win <b data-source="summary" data-field="low_death_wr_pct">—</b> of games where you die 7 times or fewer and <b data-source="summary" data-field="high_death_wr_pct">—</b> when you die 8+, and <b data-source="summary" data-field="ft_wr">—</b> when your team takes the first tower vs <b data-source="summary" data-field="nft_wr">—</b> when the enemy does. Your best champion with 5+ games is <b data-source="summary" data-field="best_champ">—</b> (<span data-source="summary" data-field="best_wr">—</span>).</p>

  <div class="kpis">
    <div class="kpi"><div class="lab">Record</div><div class="num"><span data-source="summary" data-field="wins" class="dash-skeleton">—</span>–<span data-source="summary" data-field="losses" class="dash-skeleton">—</span></div><div class="sub"><span data-source="summary" data-field="wr_pct">—</span> win rate</div></div>
    <div class="kpi"><div class="lab">Rank (latest game)</div><div class="num dash-skeleton" data-source="summary" data-field="rank_now">—</div><div class="sub">high: <span data-source="summary" data-field="rank_high">—</span></div></div>
    <div class="kpi"><div class="lab">Best champion (5+ games)</div><div class="num dash-skeleton" data-source="summary" data-field="best_champ">—</div><div class="sub"><span data-source="summary" data-field="best_rec">—</span> · everything else <span data-source="summary" data-field="rest_wr_pct">—</span></div></div>
    <div class="kpi"><div class="lab">Win rate at 0–7 deaths</div><div class="num good dash-skeleton" data-source="summary" data-field="low_death_wr_pct">—</div><div class="sub">at 8+ deaths: <span class="bad" data-source="summary" data-field="high_death_wr_pct">—</span></div></div>
  </div>

  <div class="card" id="ladder-card">
    <h3>Rank over the period</h3>
    <p class="note">Your rank as recorded with each ranked game, in game order. Gridlines mark divisions.</p>
    <div class="chart" id="ladder" data-source="ranked_games"></div>
  </div>

  <h2 id="what-wins">What wins and loses your games</h2>
  <div class="grid">
    <div class="card">
      <h3>Win rate by deaths</h3>
      <p class="note">Games grouped by how many times you died. The dashed line is 50%.</p>
      <div class="chart" id="deaths" data-source="by_deaths"></div>
    </div>
    <div class="card">
      <h3>First objective taken → win rate</h3>
      <p class="note">Your win rate when your team got each objective first vs. when the enemy did.</p>
      <div class="legend"><span><i style="background:var(--color-ok)"></i>Your team first</span><span><i style="background:var(--cds-chart-muted)"></i>Enemy first</span></div>
      <div class="chart" id="objectives" data-source="by_objective"></div>
    </div>
  </div>

  <h2 id="timing">How games play out over time</h2>
  <div class="grid">
    <div class="card">
      <h3>Your OP Score through the game</h3>
      <p class="note">OP.GG's running performance score, averaged at each minute, in wins vs losses.</p>
      <div class="legend"><span><i style="background:var(--color-ok)"></i>Wins</span><span><i style="background:var(--color-bad)"></i>Losses</span></div>
      <div class="chart" id="opcurve" data-source="op_curve"></div>
    </div>
    <div class="card">
      <h3>Win rate by game length</h3>
      <p class="note">Win rate by how long the game lasted.</p>
      <div class="chart" id="duration" data-source="by_duration"></div>
    </div>
  </div>

  <h2 id="champs">Roles and champions</h2>
  <div class="grid">
    <div class="card">
      <h3>Win rate by role</h3>
      <p class="note">Bar = win rate. "Ahead in lane" = share of games you ended with more gold than your lane opponent.</p>
      <div class="chart" id="roles" data-source="by_role"></div>
    </div>
    <div class="card">
      <h3>Lane result → win rate</h3>
      <p class="note">Win rate when you ended with more gold than your direct opponent vs. less.</p>
      <div class="legend"><span><i style="background:var(--color-ok)"></i>Ahead of opponent</span><span><i style="background:var(--cds-chart-muted)"></i>Behind</span></div>
      <div class="chart" id="lane" data-source="by_lane"></div>
    </div>
  </div>
  <div class="card" style="margin-top:var(--cds-gap-md)">
    <h3>Champions (3+ games)</h3>
    <p class="note">Everything with fewer games is grouped as "All others". KP = kill participation.</p>
    <div class="tblwrap">
      <table id="champ-table" data-source="by_champ" data-row-key="champion">
        <thead><tr>
          <th data-field="champion">Champion</th><th data-field="games">Games</th><th data-field="wr">Win rate</th>
          <th data-field="kda">K / D / A</th><th class="hide-narrow" data-field="cspm">CS/min</th>
          <th class="hide-narrow" data-field="kp">KP</th><th class="hide-narrow" data-field="minutes">Avg min</th>
        </tr></thead><tbody></tbody>
      </table>
    </div>
  </div>

  <h2 id="when">When you play</h2>
  <div class="grid">
    <div class="card">
      <h3>Win rate by start time</h3>
      <p class="note">Local start time, in the time zone set in the notebook.</p>
      <div class="chart" id="hours" data-source="by_hour"></div>
    </div>
    <div class="card">
      <h3>Win rate by game # in a session</h3>
      <p class="note">A session ends after a 90+ minute gap. Small samples past game 3.</p>
      <div class="chart" id="session" data-source="by_session_game"></div>
    </div>
  </div>

  <h2 id="minute">Minute by minute</h2>
  <p class="take">From your Riot timeline export: <span data-source="tl_summary" data-field="games" class="dash-skeleton">—</span> ranked solo games, <span data-source="tl_summary" data-field="first_date" class="dash-skeleton">—</span> to <span data-source="tl_summary" data-field="last_date" class="dash-skeleton">—</span> (remakes removed). Gold, deaths and objectives minute by minute.</p>
  <div class="kpis">
    <div class="kpi"><div class="lab">Team gold at 15 · losses</div><div class="num bad dash-skeleton" data-source="tl_summary" data-field="tg15_loss">—</div><div class="sub">in wins: <span data-source="tl_summary" data-field="tg15_win">—</span></div></div>
    <div class="kpi"><div class="lab">First death (median)</div><div class="num dash-skeleton" data-source="tl_summary" data-field="fd_all">—</div><div class="sub"><span data-source="tl_summary" data-field="fd_main_role">—</span>: <span data-source="tl_summary" data-field="fd_main">—</span></div></div>
    <div class="kpi"><div class="lab">Deaths after 14 min · per game</div><div class="num dash-skeleton" data-source="tl_summary" data-field="late_d_loss">—</div><div class="sub">in losses · wins <span data-source="tl_summary" data-field="late_d_win">—</span></div></div>
    <div class="kpi"><div class="lab">Won fights → objective</div><div class="num dash-skeleton" data-source="tl_summary" data-field="conv_all">—</div><div class="sub">wins <span data-source="tl_summary" data-field="conv_win">—</span> · losses <span data-source="tl_summary" data-field="conv_loss">—</span></div></div>
    <div class="kpi"><div class="lab">Your deaths after 14 min that gave up an objective</div><div class="num bad dash-skeleton" data-source="tl_summary" data-field="gave_obj">—</div><div class="sub">enemy took a tower, dragon, herald or baron within 60s</div></div>
  </div>
  <div class="grid">
    <div class="card">
      <h3>Team gold lead by minute</h3>
      <p class="note">Your team's gold minus the enemy team's, averaged across games still running at that minute.</p>
      <div class="legend"><span><i style="background:var(--color-ok)"></i>Wins</span><span><i style="background:var(--color-bad)"></i>Losses</span></div>
      <div class="chart tall" id="tl-team" data-source="tl_team_curve"></div>
    </div>
    <div class="card">
      <h3>Win rate by team gold at 15</h3>
      <p class="note">Games grouped by your team's gold lead at 15 minutes.</p>
      <div class="chart" id="tl-team15" data-source="tl_team15"></div>
    </div>
    <div class="card">
      <h3>Your gold vs your lane opponent</h3>
      <p class="note">Average gold difference against the player in the same role, by minute. Roles with 5+ games.</p>
      <div class="legend" id="tl-lane-legend"></div>
      <div class="chart tall" id="tl-lane" data-source="tl_lane_curve"></div>
    </div>
    <div class="card">
      <h3>Your deaths per game, by phase</h3>
      <p class="note">Average deaths per game before 14 minutes, 14–25 and after 25, wins vs losses.</p>
      <div class="legend"><span><i style="background:var(--color-ok)"></i>Wins</span><span><i style="background:var(--color-bad)"></i>Losses</span></div>
      <div class="chart" id="tl-phase" data-source="tl_phase"></div>
    </div>
  </div>
  <div class="grid" style="margin-top:var(--cds-gap-md)">
    <div class="card">
      <h3>How you die early, by role</h3>
      <p class="note">Before 14 minutes. "Solo" = killed by one enemy alone; otherwise two or more enemies were involved.</p>
      <div class="tblwrap">
        <table id="tl-roles" data-source="tl_role_deaths" data-row-key="role">
          <thead><tr><th data-field="role">Role</th><th data-field="games">Games</th><th data-field="first_death">First death (median)</th><th data-field="early_per_game">Deaths before 14</th><th data-field="solo_share">Solo</th></tr></thead><tbody></tbody>
        </table>
      </div>
    </div>
    <div class="card">
      <h3>Where you die, by role</h3>
      <p class="note">Share of each role's deaths by map area.</p>
      <div class="tblwrap">
        <table id="tl-zones" data-source="tl_zones" data-row-key="zone">
          <thead><tr><th data-field="zone">Area</th><th data-field="Top">Top</th><th data-field="Jungle">Jungle</th><th data-field="Mid">Mid</th><th data-field="ADC">ADC</th><th data-field="Support">Support</th></tr></thead><tbody></tbody>
        </table>
      </div>
    </div>
  </div>

  <h2 id="objectives">Objectives: showing up and staying alive</h2>
  <p class="take">Every dragon, grubs, herald, Baron and Elder in your <span data-source="obj_summary" data-field="games">—</span> timeline games (<span data-source="obj_summary" data-field="objectives">—</span> objectives). "Near" = within about a screen of the pit at the last minute mark before it fell (Riot records positions once a minute).</p>
  <div class="kpis">
    <div class="kpi"><div class="lab">Your team takes it · you near</div><div class="num good dash-skeleton" data-source="obj_summary" data-field="take_near">—</div><div class="sub">you away: <span data-source="obj_summary" data-field="take_away">—</span></div></div>
    <div class="kpi"><div class="lab">Your team takes it · 0 team deaths before</div><div class="num good dash-skeleton" data-source="obj_summary" data-field="take_0d">—</div><div class="sub">3+ deaths before: <span class="bad" data-source="obj_summary" data-field="take_3d">—</span></div></div>
    <div class="kpi"><div class="lab">Objectives you died just before</div><div class="num bad dash-skeleton" data-source="obj_summary" data-field="died_before">—</div><div class="sub">your team took those: <span data-source="obj_summary" data-field="take_died">—</span></div></div>
    <div class="kpi"><div class="lab">Barons you were near</div><div class="num dash-skeleton" data-source="obj_summary" data-field="baron_near">—</div><div class="sub">your team took them when near: <span data-source="obj_summary" data-field="baron_take_near">—</span></div></div>
  </div>
  <div class="grid">
    <div class="card">
      <h3>Your team's take rate: you near vs away</h3>
      <p class="note">Share of each objective your team got, split by whether you were near the pit.</p>
      <div class="legend"><span><i style="background:var(--color-ok)"></i>You near</span><span><i style="background:var(--cds-chart-muted)"></i>You away</span></div>
      <div class="chart" id="ob-near" data-source="obj_near"></div>
    </div>
    <div class="card">
      <h3>Take rate by your team's deaths just before</h3>
      <p class="note">Your team's deaths in the 90 seconds before the objective fell. Each death cuts your odds.</p>
      <div class="chart" id="ob-deaths" data-source="obj_deaths"></div>
    </div>
    <div class="card">
      <h3>How often you're there, by role</h3>
      <p class="note">Share of each objective you were near when it fell.</p>
      <div class="tblwrap">
        <table id="ob-roles" data-source="obj_roles" data-row-key="role">
          <thead><tr><th data-field="role">Role</th><th data-field="Dragon">Dragon</th><th data-field="Grubs">Grubs</th><th data-field="Herald">Herald</th><th data-field="Baron">Baron</th><th data-field="Elder">Elder</th></tr></thead><tbody></tbody>
        </table>
      </div>
    </div>
    <div class="card">
      <h3>Win rate by how often you showed up</h3>
      <p class="note">Your games split into thirds by the share of objectives you were near. Partly cause, partly effect: winning teams group more.</p>
      <div class="chart" id="ob-games" data-source="obj_games"></div>
    </div>
  </div>

  <h2 id="plan">Highlights</h2>
  <div class="card">
    <p class="note">Worked out from your own games. Small samples (under ~10 games) are left out.</p>
    <ol class="tips" id="highlights" data-source="highlights"></ol>
  </div>
  <div class="tip" id="tip"></div>
</div>

<script>
dash.calc("summary", {
  title: "Headline figures", inputs: ["ranked_games"],
  description: "Record, win rates and rank from the ranked game list; low deaths = 0–7, high = 8+; recent = last 30 games.",
  fn: (ranked_games) => {
    const g = ranked_games, won = (x) => x.win === "1", n = (xs) => xs.filter(won).length;
    const pct = (a, b) => (b ? Math.round((100 * a) / b) + "%" : "—");
    const byChamp = {}; g.forEach((x) => (byChamp[x.champion] = byChamp[x.champion] || []).push(x));
    const best = Object.entries(byChamp).filter(([, xs]) => xs.length >= 5).sort((a, b) => n(b[1]) / b[1].length - n(a[1]) / a[1].length || b[1].length - a[1].length)[0];
    const m = best ? best[1] : [], rest = g.filter((x) => !best || x.champion !== best[0]);
    const ft = g.filter((x) => x.first_tower === "1"), nft = g.filter((x) => x.first_tower !== "1");
    const last30 = g.slice(-30), lo = g.filter((x) => Number(x.deaths) <= 7), hi = g.filter((x) => Number(x.deaths) >= 8);
    const name = (r) => r.tier + " " + r.division + " · " + r.lp + " LP";
    const top = g.reduce((a, b) => (Number(b.rank_points) > Number(a.rank_points) ? b : a));
    return [{
      games: g.length, wins: n(g), losses: g.length - n(g), wr_pct: pct(n(g), g.length),
      first_date: g[0].date, last_date: g[g.length - 1].date,
      best_champ: best ? best[0] : "—", best_wr: pct(n(m), m.length), best_rec: best ? n(m) + "–" + (m.length - n(m)) + " · " + pct(n(m), m.length) : "—",
      rest_wr_pct: pct(n(rest), rest.length), ft_wr: pct(n(ft), ft.length), nft_wr: pct(n(nft), nft.length),
      low_death_wr_pct: pct(n(lo), lo.length), high_death_wr_pct: pct(n(hi), hi.length),
      rank_now: name(g[g.length - 1]), rank_high: name(top),
    }];
  },
});

dash.calc("by_deaths", {
  title: "Win rate by deaths", description: "Games grouped by your deaths: 0–4, 5–7, 8–10, 11+.", inputs: ["ranked_games"],
  fn: (ranked_games) => {
    const bin = (d) => (d <= 4 ? "0–4" : d <= 7 ? "5–7" : d <= 10 ? "8–10" : "11+");
    return ["0–4", "5–7", "8–10", "11+"].map((group) => {
      const xs = ranked_games.filter((x) => bin(Number(x.deaths)) === group);
      return { group, games: xs.length, wr: xs.length ? xs.filter((x) => x.win === "1").length / xs.length : null };
    }).filter((r) => r.games > 0);
  },
});

dash.calc("by_duration", {
  title: "Win rate by game length", description: "Games grouped by length in minutes.", inputs: ["ranked_games"],
  fn: (ranked_games) => {
    const bin = (m) => (m < 25 ? "<25 min" : m < 30 ? "25–30" : m < 35 ? "30–35" : m < 40 ? "35–40" : "40+");
    return ["<25 min", "25–30", "30–35", "35–40", "40+"].map((group) => {
      const xs = ranked_games.filter((x) => bin(Number(x.minutes)) === group);
      return { group, games: xs.length, wr: xs.length ? xs.filter((x) => x.win === "1").length / xs.length : null };
    }).filter((r) => r.games > 0);
  },
});

dash.calc("by_hour", {
  title: "Win rate by start time", description: "Start hour in US Eastern time: morning 3am–noon, afternoon, 6–11pm, 11pm–3am.", inputs: ["ranked_games"],
  fn: (ranked_games) => {
    const bin = (h) => (h >= 3 && h < 12 ? "Morning" : h >= 12 && h < 18 ? "12–6pm" : h >= 18 && h < 23 ? "6–11pm" : "11pm–3am");
    return ["Morning", "12–6pm", "6–11pm", "11pm–3am"].map((group) => {
      const xs = ranked_games.filter((x) => bin(Number(x.hour_et)) === group);
      return { group, games: xs.length, wr: xs.length ? xs.filter((x) => x.win === "1").length / xs.length : null };
    }).filter((r) => r.games > 0);
  },
});

dash.calc("by_session_game", {
  title: "Win rate by game # in session", description: "Position of the game in a session (a 90+ minute gap starts a new one).", inputs: ["ranked_games"],
  fn: (ranked_games) => {
    const bin = (k) => (k >= 4 ? "4th+" : ["", "1st", "2nd", "3rd"][k]);
    return ["1st", "2nd", "3rd", "4th+"].map((group) => {
      const xs = ranked_games.filter((x) => bin(Number(x.session_game)) === group);
      return { group, games: xs.length, wr: xs.length ? xs.filter((x) => x.win === "1").length / xs.length : null };
    }).filter((r) => r.games > 0);
  },
});

dash.calc("by_role", {
  title: "Win rate by role", inputs: ["ranked_games"],
  description: "Per role: games, win rate, and share of games ending with more gold than the lane opponent.",
  fn: (ranked_games) => ["Top", "Support", "ADC", "Jungle", "Mid"].map((group) => {
    const xs = ranked_games.filter((x) => x.role === group);
    return { group, games: xs.length, wr: xs.length ? xs.filter((x) => x.win === "1").length / xs.length : null,
      ahead_share: xs.length ? xs.filter((x) => Number(x.lane_gold_diff) > 0).length / xs.length : null };
  }).filter((r) => r.games > 0),
});

dash.calc("by_lane", {
  title: "Lane result vs win rate", inputs: ["ranked_games"],
  description: "Per role with 5+ games: win rate when ending ahead vs behind the lane opponent in gold.",
  fn: (ranked_games) => ["Top", "Support", "ADC", "Jungle"].map((group) => {
    const xs = ranked_games.filter((x) => x.role === group);
    const a = xs.filter((x) => Number(x.lane_gold_diff) > 0), b = xs.filter((x) => !(Number(x.lane_gold_diff) > 0));
    const wr = (v) => (v.length ? v.filter((x) => x.win === "1").length / v.length : null);
    return { group, a: wr(a), b: wr(b), na: a.length, nb: b.length };
  }).filter((r) => r.na + r.nb >= 5),
});

dash.calc("by_objective", {
  title: "First objectives vs win rate", inputs: ["ranked_games"],
  description: "Win rate when your team took each objective first vs when it didn't.",
  fn: (ranked_games) => [["first_blood", "First blood"], ["first_dragon", "First dragon"], ["first_grubs", "First grubs"],
    ["first_herald", "First herald"], ["first_tower", "First tower"], ["first_baron", "First baron"]].map(([k, group]) => {
    const a = ranked_games.filter((x) => x[k] === "1"), b = ranked_games.filter((x) => x[k] !== "1");
    const wr = (v) => (v.length ? v.filter((x) => x.win === "1").length / v.length : null);
    return { group, a: wr(a), b: wr(b), na: a.length, nb: b.length };
  }),
});

dash.calc("op_curve", {
  title: "OP Score curve", inputs: ["ranked_games"],
  description: "Average OP Score at 5, 10, 15, 20 and 25 minutes, wins vs losses (only games still running at that minute).",
  fn: (ranked_games) => [5, 10, 15, 20, 25].map((minute) => {
    const avg = (xs) => { const v = xs.map((x) => x["op_" + minute]).filter((s) => s !== "" && s != null).map(Number);
      return v.length ? v.reduce((s, n) => s + n, 0) / v.length : null; };
    return { minute, wins: avg(ranked_games.filter((x) => x.win === "1")), losses: avg(ranked_games.filter((x) => x.win !== "1")) };
  }),
});

dash.calc("by_champ", {
  title: "Champion stats", inputs: ["ranked_games"],
  description: "Per champion with 3+ games, plus all others combined.",
  fn: (ranked_games) => {
    const mean = (xs, k) => xs.reduce((s, x) => s + Number(x[k]), 0) / xs.length;
    const counts = {}; ranked_games.forEach((x) => { counts[x.champion] = (counts[x.champion] || 0) + 1; });
    const row = (champion, xs) => ({
      champion, games: xs.length, wr: xs.filter((x) => x.win === "1").length / xs.length,
      kda: [mean(xs, "kills"), mean(xs, "deaths"), mean(xs, "assists")].map((n) => n.toFixed(1)).join(" / "),
      cspm: mean(xs, "cs_per_min"), kp: mean(xs, "kill_participation"), minutes: mean(xs, "minutes"),
    });
    const main = Object.entries(counts).filter(([, n]) => n >= 3).sort((a, b) => b[1] - a[1])
      .map(([c]) => row(c, ranked_games.filter((x) => x.champion === c)));
    const others = ranked_games.filter((x) => counts[x.champion] < 3);
    return others.length ? [...main, row("All others", others)] : main;
  },
});
</script>

<script>
const tip = document.getElementById("tip"), root = document.querySelector(".rv");
const showTip = (ev, html) => {
  tip.replaceChildren(); html.forEach(([cls, t]) => { const d = document.createElement("div"); if (cls) d.className = cls; d.textContent = t; tip.appendChild(d); });
  tip.style.display = "block";
  const rb = root.getBoundingClientRect(), tw = tip.offsetWidth;
  let x = ev.clientX - rb.left + 12; if (x + tw > rb.width) x = ev.clientX - rb.left - tw - 12;
  tip.style.left = Math.max(0, x) + "px"; tip.style.top = (ev.clientY - rb.top + 12) + "px";
};
const hideTip = () => { tip.style.display = "none"; };
const pct = d3.format(".0%");
const fail = (el) => { el.replaceChildren(); const d = document.createElement("div"); d.className = "fail"; d.textContent = "Failed to load"; el.appendChild(d); };
const muted = (s) => s.attr("fill", "var(--color-fg-muted)").attr("font-size", "var(--cds-font-size-caption)");
const textW = (svg, s) => { const t = svg.append("text").attr("font-size", "var(--cds-font-size-caption)").text(s); const w = t.node().getComputedTextLength(); t.remove(); return w; };

function hbars(el, rows, opts = {}) {
  el.replaceChildren();
  const W = el.clientWidth || 320, rowH = 30;
  const svg = d3.select(el).append("svg").attr("width", "100%");
  const left = d3.max(rows, (r) => textW(svg, r.group)) + 10, right = textW(svg, "100% · 99 games") + 8;
  const H = rows.length * rowH + 24; svg.attr("viewBox", `0 0 ${W} ${H}`);
  const x = d3.scaleLinear([0, 1], [left, Math.max(left + 40, W - right)]);
  const y = d3.scaleBand(rows.map((r) => r.group), [4, H - 20]).padding(0.3);
  [0, 0.5, 1].forEach((t) => {
    svg.append("line").attr("x1", x(t)).attr("x2", x(t)).attr("y1", 0).attr("y2", H - 18)
      .attr("stroke", t === 0.5 ? "var(--cds-chart-reference)" : "var(--cds-chart-grid)").attr("stroke-dasharray", t === 0.5 ? "4 3" : null);
    muted(svg.append("text").attr("x", x(t)).attr("y", H - 4).attr("text-anchor", "middle").text(pct(t)));
  });
  const g = svg.selectAll("g.r").data(rows).join("g");
  muted(g.append("text").attr("x", left - 8).attr("y", (r) => y(r.group) + y.bandwidth() / 2).attr("dy", "0.35em").attr("text-anchor", "end").text((r) => r.group)).attr("fill", "var(--color-fg)");
  g.append("rect").attr("x", x(0)).attr("y", (r) => y(r.group)).attr("height", Math.min(y.bandwidth(), 24)).attr("rx", 4)
    .attr("width", (r) => Math.max(2, x(r.wr ?? 0) - x(0)))
    .attr("fill", (r) => (opts.color ? opts.color(r) : r.wr >= 0.5 ? "var(--color-ok)" : "var(--cds-chart-muted)"))
    .on("mousemove", (ev, r) => showTip(ev, [["t", r.group], [null, pct(r.wr) + " win rate · " + r.games + " games" + (r.ahead_share != null ? " · ahead in lane " + pct(r.ahead_share) : "")]]))
    .on("mouseleave", hideTip);
  muted(g.append("text").attr("x", (r) => x(r.wr ?? 0) + 6).attr("y", (r) => y(r.group) + Math.min(y.bandwidth(), 24) / 2).attr("dy", "0.35em")
    .text((r) => pct(r.wr) + " · " + r.games + (r.ahead_share != null ? " · ahead " + pct(r.ahead_share) : "")));
}

function pairs(el, rows, labA, labB) {
  el.replaceChildren();
  const W = el.clientWidth || 320, rowH = 30;
  const svg = d3.select(el).append("svg").attr("width", "100%");
  const left = d3.max(rows, (r) => textW(svg, r.group)) + 10, H = rows.length * rowH + 24;
  svg.attr("viewBox", `0 0 ${W} ${H}`);
  const x = d3.scaleLinear([0, 1], [left, W - 34]);
  const y = d3.scaleBand(rows.map((r) => r.group), [4, H - 20]).padding(0.3);
  [0, 0.5, 1].forEach((t) => {
    svg.append("line").attr("x1", x(t)).attr("x2", x(t)).attr("y1", 0).attr("y2", H - 18)
      .attr("stroke", t === 0.5 ? "var(--cds-chart-reference)" : "var(--cds-chart-grid)").attr("stroke-dasharray", t === 0.5 ? "4 3" : null);
    muted(svg.append("text").attr("x", x(t)).attr("y", H - 4).attr("text-anchor", "middle").text(pct(t)));
  });
  const g = svg.selectAll("g.r").data(rows).join("g").attr("transform", (r) => `translate(0,${y(r.group) + y.bandwidth() / 2})`);
  muted(g.append("text").attr("x", left - 8).attr("dy", "0.35em").attr("text-anchor", "end").text((r) => r.group)).attr("fill", "var(--color-fg)");
  g.append("line").attr("x1", (r) => x(r.b ?? 0)).attr("x2", (r) => x(r.a ?? 0)).attr("stroke", "var(--cds-chart-grid)").attr("stroke-width", 4).attr("stroke-linecap", "round");
  const dot = (key, fill, n, lab) => g.append("circle").attr("cx", (r) => x(r[key] ?? 0)).attr("r", 6).attr("fill", fill)
    .on("mousemove", (ev, r) => showTip(ev, [["t", r.group + " · " + lab], [null, pct(r[key]) + " win rate · " + r[n] + " games"]])).on("mouseleave", hideTip);
  dot("b", "var(--cds-chart-muted)", "nb", labB); dot("a", "var(--color-ok)", "na", labA);
  muted(g.append("text").attr("x", (r) => x(Math.max(r.a ?? 0, r.b ?? 0)) + 10).attr("dy", "0.35em").text((r) => pct(r.a)));
}

function opcurve(el, rows) {
  el.replaceChildren();
  const W = el.clientWidth || 320, H = 210, m = { t: 12, r: 64, b: 26, l: 34 };
  const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
  const vals = rows.flatMap((r) => [r.wins, r.losses]).filter((v) => v != null);
  const x = d3.scaleLinear([5, 25], [m.l, W - m.r]), y = d3.scaleLinear([Math.floor(d3.min(vals) * 2) / 2 - 0.25, Math.ceil(d3.max(vals) * 2) / 2], [H - m.b, m.t]);
  y.ticks(4).forEach((t) => { svg.append("line").attr("x1", m.l).attr("x2", W - m.r).attr("y1", y(t)).attr("y2", y(t)).attr("stroke", "var(--cds-chart-grid)");
    muted(svg.append("text").attr("x", m.l - 6).attr("y", y(t)).attr("dy", "0.35em").attr("text-anchor", "end").text(t.toFixed(1))); });
  rows.forEach((r) => muted(svg.append("text").attr("x", x(r.minute)).attr("y", H - 6).attr("text-anchor", "middle").text(r.minute + "m")));
  [["wins", "var(--color-ok)", "Wins"], ["losses", "var(--color-bad)", "Losses"]].forEach(([k, c, lab]) => {
    const pts = rows.filter((r) => r[k] != null);
    svg.append("path").datum(pts).attr("fill", "none").attr("stroke", c).attr("stroke-width", 2).attr("stroke-linecap", "round").attr("stroke-linejoin", "round")
      .attr("d", d3.line((r) => x(r.minute), (r) => y(r[k])));
    svg.selectAll(null).data(pts).join("circle").attr("cx", (r) => x(r.minute)).attr("cy", (r) => y(r[k])).attr("r", 4).attr("fill", c)
      .on("mousemove", (ev, r) => showTip(ev, [["t", r.minute + " min · " + lab], [null, "OP Score " + r[k].toFixed(2)]])).on("mouseleave", hideTip);
    const last = pts[pts.length - 1];
    svg.append("text").attr("x", x(last.minute) + 8).attr("y", y(last[k])).attr("dy", "0.35em").attr("font-size", "var(--cds-font-size-caption)").attr("fill", c).text(lab + " " + last[k].toFixed(1));
  });
}

function ladder(el, g) {
  el.replaceChildren();
  const pts = g.filter((r) => !isNaN(r.rank_points));
  const W = el.clientWidth || 320, H = 200, m = { t: 10, r: 12, b: 26, l: 0 };
  const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
  const tiers = ["Iron", "Bronze", "Silver", "Gold", "Platinum", "Emerald", "Diamond", "Master"], div = ["IV", "III", "II", "I"];
  const lo = Math.floor((d3.min(pts, (r) => r.rank_points) - 20) / 100) * 100, hi = Math.ceil((d3.max(pts, (r) => r.rank_points) + 20) / 100) * 100;
  const lab = (v) => tiers[Math.floor(v / 400)] + " " + div[Math.floor((v % 400) / 100)];
  m.l = d3.max(d3.range(lo, hi, 100), (v) => textW(svg, lab(v))) + 10;
  const x = d3.scaleLinear([1, d3.max(pts, (r) => r.game)], [m.l, W - m.r]), y = d3.scaleLinear([lo, hi], [H - m.b, m.t]);
  d3.range(lo, hi + 1, 100).forEach((v) => {
    svg.append("line").attr("x1", m.l).attr("x2", W - m.r).attr("y1", y(v)).attr("y2", y(v)).attr("stroke", "var(--cds-chart-grid)");
    if (v < hi) muted(svg.append("text").attr("x", m.l - 6).attr("y", y(v + 50)).attr("dy", "0.35em").attr("text-anchor", "end").text(lab(v)));
  });
  const fmt = d3.utcFormat("%b %-d");
  const ticks = x.ticks(Math.max(2, Math.floor((W - m.l) / 110))).filter((t) => Number.isInteger(t));
  ticks.forEach((t) => { const r = pts.find((p) => p.game >= t); if (r) muted(svg.append("text").attr("x", x(t)).attr("y", H - 6).attr("text-anchor", "middle").text(fmt(new Date(r.date)))); });
  svg.append("path").datum(pts).attr("fill", "none").attr("stroke", dash.colors[0]).attr("stroke-width", 2).attr("stroke-linejoin", "round").attr("stroke-linecap", "round")
    .attr("d", d3.line((r) => x(r.game), (r) => y(r.rank_points)));
  const hit = svg.append("rect").attr("x", m.l).attr("y", m.t).attr("width", W - m.l - m.r).attr("height", H - m.t - m.b).attr("fill", "transparent");
  const dot = svg.append("circle").attr("r", 4).attr("fill", dash.colors[0]).style("display", "none");
  hit.on("mousemove", (ev) => {
    const [mx] = d3.pointer(ev); const gi = Math.round(x.invert(mx)); const r = pts.reduce((a, b) => (Math.abs(b.game - gi) < Math.abs(a.game - gi) ? b : a));
    dot.style("display", null).attr("cx", x(r.game)).attr("cy", y(r.rank_points));
    showTip(ev, [["t", fmt(new Date(r.date)) + " · game " + r.game], [null, r.tier + " " + r.division + " · " + r.lp + " LP · " + r.champion + " " + (r.win ? "win" : "loss")]]);
  }).on("mouseleave", () => { dot.style("display", "none"); hideTip(); });
}

function champTable(rows) {
  const tb = document.querySelector("#champ-table tbody"); tb.replaceChildren();
  const f1 = d3.format(".1f");
  rows.forEach((r) => {
    const tr = document.createElement("tr");
    [r.champion, r.games, pct(r.wr), r.kda, f1(r.cspm), pct(r.kp), f1(r.minutes)].forEach((v, i) => {
      const td = document.createElement("td"); td.textContent = v; if (i >= 4) td.className = "hide-narrow";
      if (i === 2) td.className = r.wr >= 0.55 ? "good" : r.wr < 0.42 ? "bad" : ""; tr.appendChild(td);
    });
    tb.appendChild(tr);
  });
}

dash.onData(() => {
  const ids = ["summary", "by_deaths", "by_objective", "op_curve", "by_duration", "by_role", "by_lane", "by_champ", "by_hour", "by_session_game"];
  const gd = dash.data("ranked_games");
  const charts = { ladder: "ranked_games", deaths: "by_deaths", objectives: "by_objective", opcurve: "op_curve", duration: "by_duration", roles: "by_role", lane: "by_lane", hours: "by_hour", session: "by_session_game" };
  Object.entries(charts).forEach(([elId, src]) => {
    const el = document.getElementById(elId), d = dash.data(src);
    if (d.status === "error" || gd.status === "error") return fail(el);
    if (d.status !== "ok") { el.replaceChildren(); return; }
    if (elId === "ladder") ladder(el, d.data.map((r) => ({ ...r, game: Number(r.game), rank_points: r.rank_points === "" ? NaN : Number(r.rank_points), win: r.win === "1" })));
    else if (elId === "objectives") pairs(el, d.data, "your team first", "enemy first");
    else if (elId === "lane") pairs(el, d.data, "ahead of opponent", "behind opponent");
    else if (elId === "opcurve") opcurve(el, d.data);
    else hbars(el, d.data);
  });
  const s = dash.data("summary");
  document.querySelectorAll('[data-source="summary"]').forEach((el) => {
    el.classList.remove("dash-skeleton");
    el.textContent = s.status === "ok" ? String(s.data[0][el.dataset.field]) : "—";
  });
  const c = dash.data("by_champ");
  if (c.status === "ok") champTable(c.data); else document.querySelector("#champ-table tbody").replaceChildren();
});
</script>

<script>
dash.calc("tl_summary", {
  title: "Timeline headline figures", inputs: ["timeline_games", "timeline_deaths", "timeline_fights"],
  description: "Team gold at 15, median first death, deaths after 14 min, fight-to-objective rate and deaths before enemy objectives.",
  fn: (timeline_games, timeline_deaths, timeline_fights) => {
    const mean = (xs) => (xs.length ? xs.reduce((s, n) => s + n, 0) / xs.length : null);
    const median = (xs) => { const v = xs.slice().sort((a, b) => a - b); const m = v.length >> 1; return v.length ? (v.length % 2 ? v[m] : (v[m - 1] + v[m]) / 2) : null; };
    const sgn = (n) => (n >= 0 ? "+" : "−") + Math.abs(Math.round(n)).toLocaleString("en-US");
    const pct = (n) => Math.round(100 * n) + "%";
    const W = timeline_games.filter((g) => g.win === "1"), L = timeline_games.filter((g) => g.win !== "1");
    const tg = (xs) => mean(xs.filter((g) => g.team_gold_15 !== "").map((g) => Number(g.team_gold_15)));
    const fd = (role) => median(timeline_games.filter((g) => (!role || g.role === role) && g.first_death_min !== "").map((g) => Number(g.first_death_min)));
    const roleCount = {}; timeline_games.forEach((g) => { roleCount[g.role] = (roleCount[g.role] || 0) + 1; });
    const mainRole = Object.entries(roleCount).sort((a, b) => b[1] - a[1])[0]?.[0];
    const mins = (v) => (v == null ? "—" : v.toFixed(1) + " min");
    const late = (xs) => mean(xs.map((g) => Number(g.deaths_14_25) + Number(g.deaths_25_plus)));
    const conv = (xs) => mean(xs.map((f) => Number(f.converted)));
    const lateD = timeline_deaths.filter((x) => Number(x.minute) >= 14);
    const dates = timeline_games.map((g) => g.date).sort();
    return [{
      games: timeline_games.length, first_date: dates[0], last_date: dates[dates.length - 1],
      tg15_loss: sgn(tg(L)), tg15_win: sgn(tg(W)),
      fd_all: mins(fd()), fd_main_role: mainRole ? "as " + mainRole : "—", fd_main: mins(fd(mainRole)),
      late_d_loss: late(L).toFixed(1), late_d_win: late(W).toFixed(1),
      conv_all: pct(conv(timeline_fights)), conv_win: pct(conv(timeline_fights.filter((f) => f.win === "1"))),
      conv_loss: pct(conv(timeline_fights.filter((f) => f.win !== "1"))),
      gave_obj: pct(mean(lateD.map((x) => Number(x.enemy_objective_within_60s)))),
    }];
  },
});

dash.calc("tl_team_curve", {
  title: "Team gold lead by minute", inputs: ["timeline_minutes"],
  description: "Average team gold difference per minute (1–30), wins vs losses; minutes with fewer than 20 games on either side are left out.",
  fn: (timeline_minutes) => {
    const out = [];
    for (let m = 1; m <= 30; m++) {
      const at = timeline_minutes.filter((r) => Number(r.minute) === m);
      const w = at.filter((r) => r.win === "1").map((r) => Number(r.team_gold_diff)), l = at.filter((r) => r.win !== "1").map((r) => Number(r.team_gold_diff));
      if (w.length >= 20 && l.length >= 20) out.push({ minute: m, wins: w.reduce((s, n) => s + n, 0) / w.length, losses: l.reduce((s, n) => s + n, 0) / l.length });
    }
    return out;
  },
});

dash.calc("tl_lane_curve", {
  title: "Lane gold difference by minute", inputs: ["timeline_minutes"],
  description: "Average gold difference vs the same-role opponent per minute (1–20), for each role with 5+ games.",
  fn: (timeline_minutes) => {
    const out = [], games = {};
    timeline_minutes.forEach((r) => { (games[r.role] = games[r.role] || new Set()).add(r.match_id); });
    const roles = ["Top", "Jungle", "Mid", "ADC", "Support"].filter((r) => games[r] && games[r].size >= 5);
    for (let m = 1; m <= 20; m++) {
      const row = { minute: m };
      roles.forEach((role) => {
        const v = timeline_minutes.filter((r) => Number(r.minute) === m && r.role === role && r.lane_gold_diff !== "").map((r) => Number(r.lane_gold_diff));
        row[role] = v.length ? v.reduce((s, n) => s + n, 0) / v.length : null;
      });
      out.push(row);
    }
    return out;
  },
});

dash.calc("tl_team15", {
  title: "Win rate by team gold at 15", inputs: ["timeline_games"],
  description: "Games grouped by team gold difference at 15 minutes.",
  fn: (timeline_games) => {
    const bin = (n) => (n <= -3000 ? "Behind 3k+" : n <= -1000 ? "Behind 1–3k" : n < 1000 ? "Even ±1k" : n < 3000 ? "Ahead 1–3k" : "Ahead 3k+");
    const xs = timeline_games.filter((g) => g.team_gold_15 !== "");
    return ["Behind 3k+", "Behind 1–3k", "Even ±1k", "Ahead 1–3k", "Ahead 3k+"].map((group) => {
      const v = xs.filter((g) => bin(Number(g.team_gold_15)) === group);
      return { group, games: v.length, wr: v.length ? v.filter((g) => g.win === "1").length / v.length : null };
    }).filter((r) => r.games > 0);
  },
});

dash.calc("tl_phase", {
  title: "Deaths per game by phase", inputs: ["timeline_games"],
  description: "Your average deaths per game before 14 min, 14–25 and after 25, wins vs losses.",
  fn: (timeline_games) => [["deaths_0_14", "0–14 min"], ["deaths_14_25", "14–25 min"], ["deaths_25_plus", "25+ min"]].map(([k, group]) => {
    const avg = (xs) => xs.reduce((s, g) => s + Number(g[k]), 0) / xs.length;
    const W = timeline_games.filter((g) => g.win === "1"), L = timeline_games.filter((g) => g.win !== "1");
    return { group, a: avg(W), b: avg(L), na: W.length, nb: L.length };
  }),
});

dash.calc("tl_role_deaths", {
  title: "Early deaths by role", inputs: ["timeline_games", "timeline_deaths"],
  description: "Per role (5+ games): median first-death minute, deaths before 14 min per game, and share of those deaths with a single enemy involved.",
  fn: (timeline_games, timeline_deaths) => ["Top", "Jungle", "Mid", "ADC", "Support"].map((role) => {
    const g = timeline_games.filter((x) => x.role === role);
    const fd = g.filter((x) => x.first_death_min !== "").map((x) => Number(x.first_death_min)).sort((a, b) => a - b);
    const m = fd.length >> 1, med = fd.length ? (fd.length % 2 ? fd[m] : (fd[m - 1] + fd[m]) / 2) : null;
    const early = timeline_deaths.filter((d) => d.role === role && Number(d.minute) < 14);
    return { role, games: g.length, first_death: med, early_per_game: g.length ? early.length / g.length : null,
      solo_share: early.length ? early.filter((d) => d.attackers === "1").length / early.length : null };
  }).filter((r) => r.games >= 5),
});

dash.calc("tl_zones", {
  title: "Where you die, by role", inputs: ["timeline_deaths"],
  description: "Share of each role's deaths by map area (from death coordinates).",
  fn: (timeline_deaths) => ["Top lane", "Mid lane", "Bot lane", "River", "Own jungle", "Enemy jungle", "Base"].map((zone) => {
    const row = { zone };
    ["Top", "Jungle", "Mid", "ADC", "Support"].forEach((role) => {
      const all = timeline_deaths.filter((d) => d.role === role);
      row[role] = all.length ? all.filter((d) => d.zone === zone).length / all.length : null;
    });
    return row;
  }),
});
</script>

<script>
(() => {
  const tip = document.getElementById("tip"), root = document.querySelector(".rv");
  const show = (ev, lines) => {
    tip.replaceChildren(); lines.forEach(([c, t]) => { const d = document.createElement("div"); if (c) d.className = c; d.textContent = t; tip.appendChild(d); });
    tip.style.display = "block"; const rb = root.getBoundingClientRect(), tw = tip.offsetWidth;
    let x = ev.clientX - rb.left + 12; if (x + tw > rb.width) x = ev.clientX - rb.left - tw - 12;
    tip.style.left = Math.max(0, x) + "px"; tip.style.top = (ev.clientY - rb.top + 12) + "px";
  };
  const hide = () => { tip.style.display = "none"; };
  const pct = d3.format(".0%"), gold = (n) => (n >= 0 ? "+" : "−") + d3.format(",")(Math.abs(Math.round(n)));
  const cap = (s) => s.attr("fill", "var(--color-fg-muted)").attr("font-size", "var(--cds-font-size-caption)");
  const tw = (svg, s) => { const t = svg.append("text").attr("font-size", "var(--cds-font-size-caption)").text(s); const w = t.node().getComputedTextLength(); t.remove(); return w; };
  const fail = (el) => { el.replaceChildren(); const d = document.createElement("div"); d.className = "fail"; d.textContent = "Failed to load"; el.appendChild(d); };

  function lines(el, rows, series, xMax) {
    el.replaceChildren();
    const W = el.clientWidth || 320, H = 240;
    const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
    const vals = rows.flatMap((r) => series.map((s) => r[s.key])).filter((v) => v != null);
    const y = d3.scaleLinear([Math.min(0, d3.min(vals)), Math.max(0, d3.max(vals))], [H - 24, 10]).nice();
    const m = { l: d3.max(y.ticks(5), (t) => tw(svg, gold(t))) + 8, r: d3.max(series, (s) => tw(svg, s.label + " " + gold(rows[rows.length - 1][s.key] ?? 0))) + 10 };
    const x = d3.scaleLinear([1, xMax], [m.l, W - m.r]);
    y.ticks(5).forEach((t) => {
      svg.append("line").attr("x1", m.l).attr("x2", W - m.r).attr("y1", y(t)).attr("y2", y(t)).attr("stroke", t === 0 ? "var(--cds-chart-axis)" : "var(--cds-chart-grid)");
      cap(svg.append("text").attr("x", m.l - 6).attr("y", y(t)).attr("dy", "0.35em").attr("text-anchor", "end").text(t === 0 ? "0" : gold(t)));
    });
    x.ticks(Math.min(6, Math.floor((W - m.l - m.r) / 50))).forEach((t) => cap(svg.append("text").attr("x", x(t)).attr("y", H - 6).attr("text-anchor", "middle").text(t + "m")));
    const ends = [];
    series.forEach((s) => {
      const pts = rows.filter((r) => r[s.key] != null);
      svg.append("path").datum(pts).attr("fill", "none").attr("stroke", s.color).attr("stroke-width", 2).attr("stroke-linecap", "round").attr("stroke-linejoin", "round")
        .attr("d", d3.line((r) => x(r.minute), (r) => y(r[s.key])));
      const last = pts[pts.length - 1]; ends.push({ s, y: y(last[s.key]), v: last[s.key] });
    });
    ends.sort((a, b) => a.y - b.y); for (let i = 1; i < ends.length; i++) if (ends[i].y - ends[i - 1].y < 14) ends[i].y = ends[i - 1].y + 14;
    ends.forEach((e) => svg.append("text").attr("x", W - m.r + 6).attr("y", e.y).attr("dy", "0.35em").attr("font-size", "var(--cds-font-size-caption)").attr("fill", e.s.color).text(e.s.label + " " + gold(e.v)));
    const rule = svg.append("line").attr("y1", 10).attr("y2", H - 24).attr("stroke", "var(--cds-chart-reference)").style("display", "none");
    svg.append("rect").attr("x", m.l).attr("y", 0).attr("width", W - m.l - m.r).attr("height", H - 24).attr("fill", "transparent")
      .on("mousemove", (ev) => {
        const mn = Math.round(x.invert(d3.pointer(ev)[0])); const r = rows.find((q) => q.minute === mn); if (!r) return hide();
        rule.style("display", null).attr("x1", x(mn)).attr("x2", x(mn));
        show(ev, [["t", "Minute " + mn], ...series.filter((s) => r[s.key] != null).map((s) => [null, s.label + ": " + gold(r[s.key])])]);
      }).on("mouseleave", () => { rule.style("display", "none"); hide(); });
  }

  function hbar(el, rows) {
    el.replaceChildren();
    const W = el.clientWidth || 320, H = rows.length * 30 + 24;
    const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
    const left = d3.max(rows, (r) => tw(svg, r.group)) + 10, right = tw(svg, "100% · 99") + 8;
    const x = d3.scaleLinear([0, 1], [left, W - right]), y = d3.scaleBand(rows.map((r) => r.group), [4, H - 20]).padding(0.3);
    [0, 0.5, 1].forEach((t) => {
      svg.append("line").attr("x1", x(t)).attr("x2", x(t)).attr("y1", 0).attr("y2", H - 18).attr("stroke", t === 0.5 ? "var(--cds-chart-reference)" : "var(--cds-chart-grid)").attr("stroke-dasharray", t === 0.5 ? "4 3" : null);
      cap(svg.append("text").attr("x", x(t)).attr("y", H - 4).attr("text-anchor", "middle").text(pct(t)));
    });
    const g = svg.selectAll(null).data(rows).join("g");
    g.append("text").attr("x", left - 8).attr("y", (r) => y(r.group) + y.bandwidth() / 2).attr("dy", "0.35em").attr("text-anchor", "end").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg)").text((r) => r.group);
    g.append("rect").attr("x", x(0)).attr("y", (r) => y(r.group)).attr("height", Math.min(24, y.bandwidth())).attr("rx", 4).attr("width", (r) => Math.max(2, x(r.wr) - x(0)))
      .attr("fill", (r) => (r.wr >= 0.5 ? "var(--color-ok)" : "var(--cds-chart-muted)"))
      .on("mousemove", (ev, r) => show(ev, [["t", r.group], [null, pct(r.wr) + " win rate · " + r.games + " games"]])).on("mouseleave", hide);
    cap(g.append("text").attr("x", (r) => x(r.wr) + 6).attr("y", (r) => y(r.group) + Math.min(24, y.bandwidth()) / 2).attr("dy", "0.35em").text((r) => pct(r.wr) + " · " + r.games));
  }

  function phase(el, rows) {
    el.replaceChildren();
    const W = el.clientWidth || 320, H = rows.length * 44 + 24;
    const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
    const left = d3.max(rows, (r) => tw(svg, r.group)) + 10;
    const x = d3.scaleLinear([0, Math.ceil(d3.max(rows, (r) => Math.max(r.a, r.b)))], [left, W - 40]), y = d3.scaleBand(rows.map((r) => r.group), [4, H - 20]).padding(0.2);
    x.ticks(4).forEach((t) => { svg.append("line").attr("x1", x(t)).attr("x2", x(t)).attr("y1", 0).attr("y2", H - 18).attr("stroke", "var(--cds-chart-grid)");
      cap(svg.append("text").attr("x", x(t)).attr("y", H - 4).attr("text-anchor", "middle").text(t)); });
    const bh = Math.min(14, y.bandwidth() / 2 - 2);
    rows.forEach((r) => {
      svg.append("text").attr("x", left - 8).attr("y", y(r.group) + y.bandwidth() / 2).attr("dy", "0.35em").attr("text-anchor", "end").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg)").text(r.group);
      [["a", "var(--color-ok)", "Wins", 0], ["b", "var(--color-bad)", "Losses", 1]].forEach(([k, c, lab, i]) => {
        const yy = y(r.group) + y.bandwidth() / 2 - bh - 1 + i * (bh + 2);
        svg.append("rect").attr("x", x(0)).attr("y", yy).attr("height", bh).attr("rx", 4).attr("width", Math.max(2, x(r[k]) - x(0))).attr("fill", c)
          .on("mousemove", (ev) => show(ev, [["t", r.group + " · " + lab], [null, r[k].toFixed(2) + " deaths per game"]])).on("mouseleave", hide);
        cap(svg.append("text").attr("x", x(r[k]) + 5).attr("y", yy + bh / 2).attr("dy", "0.35em").text(r[k].toFixed(1)));
      });
    });
  }

  function table(sel, rows, fmts) {
    const tb = document.querySelector(sel + " tbody"); tb.replaceChildren();
    rows.forEach((r) => { const tr = document.createElement("tr"); fmts.forEach((f) => { const td = document.createElement("td"); td.textContent = f(r); tr.appendChild(td); }); tb.appendChild(tr); });
  }

  const roleColor = { Top: dash.colors[0], Support: dash.colors[1], ADC: dash.colors[2], Jungle: dash.colors[3], Mid: dash.colors[4] };
  const lg = document.getElementById("tl-lane-legend");
  const laneSeriesFor = (rows) => {
    const keys = ["Top", "Jungle", "Mid", "ADC", "Support"].filter((k) => rows.some((r) => r[k] != null));
    lg.replaceChildren();
    keys.forEach((k) => { const sp = document.createElement("span"); const i = document.createElement("i"); i.style.background = roleColor[k]; sp.appendChild(i); sp.appendChild(document.createTextNode(k)); lg.appendChild(sp); });
    return keys.map((k) => ({ key: k, label: k, color: roleColor[k] }));
  };

  dash.onData(() => {
    const charts = [
      ["tl-team", "tl_team_curve", (el, d) => lines(el, d, [{ key: "wins", label: "Wins", color: "var(--color-ok)" }, { key: "losses", label: "Losses", color: "var(--color-bad)" }], 30)],
      ["tl-team15", "tl_team15", hbar],
      ["tl-lane", "tl_lane_curve", (el, d) => { const s = laneSeriesFor(d); if (s.length) lines(el, d, s, 20); else el.replaceChildren(); }],
      ["tl-phase", "tl_phase", phase],
    ];
    charts.forEach(([id, src, draw]) => {
      const el = document.getElementById(id), d = dash.data(src);
      if (d.status === "error") return fail(el);
      if (d.status !== "ok" || !d.data.length) { el.replaceChildren(); return; }
      draw(el, d.data);
    });
    const s = dash.data("tl_summary");
    document.querySelectorAll('[data-source="tl_summary"]').forEach((el) => { el.classList.remove("dash-skeleton"); el.textContent = s.status === "ok" ? String(s.data[0][el.dataset.field]) : "—"; });
    const rd = dash.data("tl_role_deaths");
    if (rd.status === "ok") table("#tl-roles", rd.data, [(r) => r.role, (r) => r.games, (r) => (r.first_death == null ? "—" : r.first_death.toFixed(1) + " min"), (r) => r.early_per_game.toFixed(1), (r) => (r.solo_share == null ? "—" : pct(r.solo_share))]);
    const z = dash.data("tl_zones");
    const zp = (v) => (v == null ? "—" : pct(v));
    if (z.status === "ok") table("#tl-zones", z.data, [(r) => r.zone, (r) => zp(r.Top), (r) => zp(r.Jungle), (r) => zp(r.Mid), (r) => zp(r.ADC), (r) => zp(r.Support)]);
  });
})();
</script>
<script>
dash.calc("obj_summary", {
  title: "Objective headline figures", inputs: ["objectives"],
  description: "Take rates near vs away, by team deaths in the 90s before, objectives you died just before, and Baron presence.",
  fn: (objectives) => {
    const pct = (xs) => (xs.length ? Math.round((100 * xs.filter((o) => o.your_team_took_it === "1").length) / xs.length) + "%" : "—");
    const share = (xs, f) => (xs.length ? Math.round((100 * xs.filter(f).length) / xs.length) + "%" : "—");
    const near = objectives.filter((o) => o.you_near === "1"), away = objectives.filter((o) => o.you_near !== "1");
    const d0 = objectives.filter((o) => Number(o.team_deaths_90s_before) === 0), d3 = objectives.filter((o) => Number(o.team_deaths_90s_before) >= 3);
    const died = objectives.filter((o) => o.you_died_90s_before === "1"), baron = objectives.filter((o) => o.objective === "Baron");
    return [{ games: new Set(objectives.map((o) => o.match_id)).size, objectives: objectives.length,
      take_near: pct(near), take_away: pct(away), take_0d: pct(d0), take_3d: pct(d3),
      died_before: share(objectives, (o) => o.you_died_90s_before === "1"), take_died: pct(died),
      baron_near: share(baron, (o) => o.you_near === "1"), baron_take_near: pct(baron.filter((o) => o.you_near === "1")) }];
  },
});

dash.calc("obj_near", {
  title: "Take rate near vs away", inputs: ["objectives"],
  description: "Per objective type: share your team took when you were near the pit vs away.",
  fn: (objectives) => ["Dragon", "Grubs", "Herald", "Baron", "Elder"].map((group) => {
    const xs = objectives.filter((o) => o.objective === group);
    const a = xs.filter((o) => o.you_near === "1"), b = xs.filter((o) => o.you_near !== "1");
    const wr = (v) => (v.length ? v.filter((o) => o.your_team_took_it === "1").length / v.length : null);
    return { group, a: wr(a), b: wr(b), na: a.length, nb: b.length };
  }),
});

dash.calc("obj_deaths", {
  title: "Take rate by team deaths before", inputs: ["objectives"],
  description: "Share of objectives your team took, by how many of your team died in the 90s before it fell.",
  fn: (objectives) => ["0 deaths", "1 death", "2 deaths", "3+ deaths"].map((group, i) => {
    const xs = objectives.filter((o) => Math.min(3, Number(o.team_deaths_90s_before)) === i);
    return { group, games: xs.length, wr: xs.length ? xs.filter((o) => o.your_team_took_it === "1").length / xs.length : null };
  }),
});

dash.calc("obj_roles", {
  title: "Presence by role", inputs: ["objectives"],
  description: "Share of each objective type you were near when it fell, by the role you played (roles with 5+ games).",
  fn: (objectives) => ["Top", "Jungle", "Mid", "ADC", "Support"].filter((role) => objectives.some((o) => o.role === role)).map((role) => {
    const row = { role };
    ["Dragon", "Grubs", "Herald", "Baron", "Elder"].forEach((k) => {
      const xs = objectives.filter((o) => o.role === role && o.objective === k);
      row[k] = xs.length ? xs.filter((o) => o.you_near === "1").length / xs.length : null;
    });
    return row;
  }),
});

dash.calc("obj_games", {
  title: "Win rate by presence", inputs: ["objectives"],
  description: "Games ranked by the share of objectives you were near, split into thirds; win rate of each third.",
  fn: (objectives) => {
    const by = {};
    objectives.forEach((o) => { (by[o.match_id] = by[o.match_id] || { won: o.game_won === "1", n: 0, near: 0 }); by[o.match_id].n++; if (o.you_near === "1") by[o.match_id].near++; });
    const games = Object.values(by).map((g) => ({ won: g.won, share: g.near / g.n })).sort((a, b) => a.share - b.share);
    const third = Math.ceil(games.length / 3);
    return ["Fewest objectives", "Middle third", "Most objectives"].map((group, i) => {
      const xs = games.slice(i * third, (i + 1) * third);
      return { group, games: xs.length, wr: xs.filter((g) => g.won).length / xs.length, avg_share: xs.reduce((s, g) => s + g.share, 0) / xs.length };
    });
  },
});
</script>

<script>
(() => {
  const tip = document.getElementById("tip"), root = document.querySelector(".rv");
  const show = (ev, lines) => { tip.replaceChildren(); lines.forEach(([c, t]) => { const d = document.createElement("div"); if (c) d.className = c; d.textContent = t; tip.appendChild(d); });
    tip.style.display = "block"; const rb = root.getBoundingClientRect(), w = tip.offsetWidth; let x = ev.clientX - rb.left + 12; if (x + w > rb.width) x = ev.clientX - rb.left - w - 12;
    tip.style.left = Math.max(0, x) + "px"; tip.style.top = (ev.clientY - rb.top + 12) + "px"; };
  const hide = () => { tip.style.display = "none"; };
  const pct = d3.format(".0%");
  const cap = (s) => s.attr("fill", "var(--color-fg-muted)").attr("font-size", "var(--cds-font-size-caption)");
  const tw = (svg, s) => { const t = svg.append("text").attr("font-size", "var(--cds-font-size-caption)").text(s); const w = t.node().getComputedTextLength(); t.remove(); return w; };
  const fail = (el) => { el.replaceChildren(); const d = document.createElement("div"); d.className = "fail"; d.textContent = "Failed to load"; el.appendChild(d); };
  const axis = (svg, x, H) => [0, 0.5, 1].forEach((t) => {
    svg.append("line").attr("x1", x(t)).attr("x2", x(t)).attr("y1", 0).attr("y2", H - 18).attr("stroke", t === 0.5 ? "var(--cds-chart-reference)" : "var(--cds-chart-grid)").attr("stroke-dasharray", t === 0.5 ? "4 3" : null);
    cap(svg.append("text").attr("x", x(t)).attr("y", H - 4).attr("text-anchor", "middle").text(pct(t)));
  });

  function bars(el, rows) {
    el.replaceChildren();
    const W = el.clientWidth || 320, H = rows.length * 30 + 24;
    const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
    const left = d3.max(rows, (r) => tw(svg, r.group)) + 10, right = tw(svg, "100% · 999") + 8;
    const x = d3.scaleLinear([0, 1], [left, W - right]), y = d3.scaleBand(rows.map((r) => r.group), [4, H - 20]).padding(0.3);
    axis(svg, x, H);
    rows.forEach((r) => {
      svg.append("text").attr("x", left - 8).attr("y", y(r.group) + y.bandwidth() / 2).attr("dy", "0.35em").attr("text-anchor", "end").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg)").text(r.group);
      const bh = Math.min(24, y.bandwidth());
      svg.append("rect").attr("x", x(0)).attr("y", y(r.group)).attr("height", bh).attr("rx", 4).attr("width", Math.max(2, x(r.wr) - x(0))).attr("fill", r.wr >= 0.5 ? "var(--color-ok)" : "var(--cds-chart-muted)")
        .on("mousemove", (ev) => show(ev, [["t", r.group], [null, pct(r.wr) + " · " + r.games + (r.avg_share != null ? " games · near " + pct(r.avg_share) + " of objectives" : " objectives")]])).on("mouseleave", hide);
      cap(svg.append("text").attr("x", x(r.wr) + 6).attr("y", y(r.group) + bh / 2).attr("dy", "0.35em").text(pct(r.wr) + " · " + r.games));
    });
  }

  function pairs(el, rows) {
    el.replaceChildren();
    const W = el.clientWidth || 320, H = rows.length * 30 + 24;
    const svg = d3.select(el).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
    const left = d3.max(rows, (r) => tw(svg, r.group)) + 10;
    const x = d3.scaleLinear([0, 1], [left, W - 34]), y = d3.scaleBand(rows.map((r) => r.group), [4, H - 20]).padding(0.3);
    axis(svg, x, H);
    rows.forEach((r) => {
      const cy = y(r.group) + y.bandwidth() / 2;
      svg.append("text").attr("x", left - 8).attr("y", cy).attr("dy", "0.35em").attr("text-anchor", "end").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg)").text(r.group);
      svg.append("line").attr("x1", x(r.b ?? 0)).attr("x2", x(r.a ?? 0)).attr("y1", cy).attr("y2", cy).attr("stroke", "var(--cds-chart-grid)").attr("stroke-width", 4).attr("stroke-linecap", "round");
      [["b", "var(--cds-chart-muted)", "nb", "you away"], ["a", "var(--color-ok)", "na", "you near"]].forEach(([k, c, n, lab]) => {
        if (r[k] == null) return;
        svg.append("circle").attr("cx", x(r[k])).attr("cy", cy).attr("r", 6).attr("fill", c)
          .on("mousemove", (ev) => show(ev, [["t", r.group + " · " + lab], [null, "Your team took " + pct(r[k]) + " of " + r[n]]])).on("mouseleave", hide);
      });
      cap(svg.append("text").attr("x", x(Math.max(r.a ?? 0, r.b ?? 0)) + 10).attr("y", cy).attr("dy", "0.35em").text(pct(r.a)));
    });
  }

  dash.onData(() => {
    [["ob-near", "obj_near", pairs], ["ob-deaths", "obj_deaths", bars], ["ob-games", "obj_games", bars]].forEach(([id, src, draw]) => {
      const el = document.getElementById(id), d = dash.data(src), raw = dash.data("objectives");
      if (raw.status === "error" || d.status === "error") return fail(el);
      if (d.status !== "ok") { el.replaceChildren(); return; }
      draw(el, d.data);
    });
    const s = dash.data("obj_summary");
    document.querySelectorAll('[data-source="obj_summary"]').forEach((el) => { el.classList.remove("dash-skeleton"); el.textContent = s.status === "ok" ? String(s.data[0][el.dataset.field]) : "—"; });
    const t = dash.data("obj_roles"), tb = document.querySelector("#ob-roles tbody"); tb.replaceChildren();
    if (t.status === "ok") t.data.forEach((r) => { const tr = document.createElement("tr");
      [r.role, ...["Dragon", "Grubs", "Herald", "Baron", "Elder"].map((k) => (r[k] == null ? "—" : pct(r[k])))].forEach((v, i) => { const td = document.createElement("td"); td.textContent = v;
        tr.appendChild(td); }); tb.appendChild(tr); });
  });
})();
</script>
<script>
dash.calc("highlights", {
  title: "Highlights", inputs: ["timeline_games", "objectives"],
  description: "Plain-language findings from your timeline games: deaths, team gold at 15, objectives, roles and champions.",
  fn: (timeline_games, objectives) => {
    const out = [], pct = (a, b) => (b ? Math.round((100 * a) / b) + "%" : "—");
    const W = (xs) => xs.filter((g) => g.win === "1").length;
    const dz = (g) => Number(g.deaths_0_14) + Number(g.deaths_14_25) + Number(g.deaths_25_plus);
    const lo = timeline_games.filter((g) => dz(g) <= 7), hi = timeline_games.filter((g) => dz(g) >= 8);
    if (lo.length >= 10 && hi.length >= 10) out.push({ title: "Deaths", text: "You win " + pct(W(lo), lo.length) + " of games with 7 or fewer deaths and " + pct(W(hi), hi.length) + " with 8 or more." });
    const tg = timeline_games.filter((g) => g.team_gold_15 !== "");
    const up = tg.filter((g) => Number(g.team_gold_15) >= 1000), dn = tg.filter((g) => Number(g.team_gold_15) <= -1000);
    if (up.length >= 10 && dn.length >= 10) out.push({ title: "The first 15 minutes", text: "When your team is 1k+ gold ahead at 15 you win " + pct(W(up), up.length) + "; 1k+ behind, " + pct(W(dn), dn.length) + "." });
    const took = (xs) => xs.filter((o) => o.your_team_took_it === "1").length;
    const o0 = objectives.filter((o) => Number(o.team_deaths_90s_before) === 0), o3 = objectives.filter((o) => Number(o.team_deaths_90s_before) >= 3);
    const died = objectives.filter((o) => o.you_died_90s_before === "1");
    if (o0.length >= 10 && o3.length >= 10) out.push({ title: "Objectives", text: "Your team takes " + pct(took(o0), o0.length) + " of objectives when nobody on your team died in the 90 seconds before, and " + pct(took(o3), o3.length) + " after 3+ deaths. You died just before " + pct(died.length, objectives.length) + " of all objectives." });
    const kinds = ["Dragon", "Grubs", "Herald", "Baron"].map((k) => { const xs = objectives.filter((o) => o.objective === k); return { k, n: xs.length, near: xs.filter((o) => o.you_near === "1").length }; }).filter((x) => x.n >= 10);
    if (kinds.length) { const low = kinds.sort((a, b) => a.near / a.n - b.near / b.n)[0]; out.push({ title: "Showing up", text: "You were near the pit for " + pct(low.near, low.n) + " of " + low.k + " takes, your lowest of any objective." }); }
    const byRole = {}; timeline_games.forEach((g) => (byRole[g.role] = byRole[g.role] || []).push(g));
    const roles = Object.entries(byRole).filter(([, xs]) => xs.length >= 10).map(([r, xs]) => ({ r, n: xs.length, w: W(xs) })).sort((a, b) => b.w / b.n - a.w / a.n);
    if (roles.length >= 2) out.push({ title: "Roles", text: "Your best role is " + roles[0].r + " (" + roles[0].w + "–" + (roles[0].n - roles[0].w) + "), your weakest " + roles[roles.length - 1].r + " (" + roles[roles.length - 1].w + "–" + (roles[roles.length - 1].n - roles[roles.length - 1].w) + ")." });
    const byChamp = {}; timeline_games.forEach((g) => (byChamp[g.champion] = byChamp[g.champion] || []).push(g));
    const champs = Object.entries(byChamp).filter(([, xs]) => xs.length >= 5).map(([c, xs]) => ({ c, n: xs.length, w: W(xs) })).sort((a, b) => b.w / b.n - a.w / a.n);
    if (champs.length) {
      const b = champs[0], worst = champs[champs.length - 1];
      let t = champs.length === 1 ? b.c + " is the only champion you've played 5+ times (" + b.w + "–" + (b.n - b.w) + ")." : b.c + " is your best champion with 5+ games (" + b.w + "–" + (b.n - b.w) + ").";
      if (champs.length > 1 && worst.w / worst.n < 0.45) t += " " + worst.c + " is costing you the most (" + worst.w + "–" + (worst.n - worst.w) + ").";
      out.push({ title: "Champions", text: t });
    }
    return out.map((x, i) => ({ n: i + 1, ...x }));
  },
});
</script>

<script>
dash.onData(() => {
  const ol = document.getElementById("highlights"), d = dash.data("highlights");
  ol.replaceChildren();
  if (d.status !== "ok") return;
  d.data.forEach((x) => { const li = document.createElement("li"); const b = document.createElement("b"); b.textContent = x.title + ". "; li.append(b, document.createTextNode(x.text)); ol.appendChild(li); });
  if (!d.data.length) { const li = document.createElement("li"); li.textContent = "Not enough games yet."; ol.appendChild(li); }
});
</script>

In [ ]:
%%writefile matchups_template.html
<style>
#dash-root { --dash-gutter: var(--cds-pad-lg); }
.cx { position: relative; font-family: var(--font-anthropic-sans); max-width: 72rem; margin-inline: auto; color: var(--color-fg); }
.cx h1 { font-family: var(--font-anthropic-serif); font-weight: 400; font-size: var(--cds-font-size-title); margin: 0 0 var(--cds-gap-xs); }
.cx .take { color: var(--color-fg-muted); font-size: var(--cds-font-size-body); margin: 0 0 var(--cds-gap-lg); max-width: 52rem; line-height: 1.5; }
.cx .bar { display: flex; flex-wrap: wrap; gap: var(--cds-gap-md); align-items: center; margin-bottom: var(--cds-gap-md); }
.cx .seg { display: inline-flex; border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); overflow: hidden; }
.cx .seg button { font: inherit; font-size: var(--cds-font-size-caption); background: transparent; color: var(--color-fg-muted); border: 0; padding: var(--cds-pad-xs) var(--cds-pad-md); cursor: pointer; }
.cx .seg.big button { font-size: var(--cds-font-size-body); }
.cx .seg button + button { border-left: 1px solid var(--color-border-line); }
.cx .seg button[aria-pressed=true] { background: var(--color-panel); color: var(--color-fg); font-weight: var(--cds-font-weight-medium); }
.cx input[type=search] { font: inherit; font-size: var(--cds-font-size-caption); padding: var(--cds-pad-xs) var(--cds-pad-sm); border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); background: var(--color-bg); color: var(--color-fg); width: 10rem; }
.cx .hint { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); }
.cx .tiles { display: grid; grid-template-columns: repeat(auto-fill, minmax(6.5rem, 1fr)); gap: var(--cds-gap-sm); }
.cx .tile { font: inherit; display: flex; flex-direction: column; align-items: center; gap: .3rem; padding: var(--cds-pad-sm) var(--cds-pad-xs); background: var(--color-panel); border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); color: var(--color-fg); cursor: pointer; min-width: 0; }
.cx .tile:hover { border-color: var(--cds-chart-axis); }
.cx .tile[aria-pressed=true] { outline: 2px solid var(--color-fg); outline-offset: -1px; }
.cx .mono { width: 2.6rem; height: 2.6rem; border-radius: 50%; display: grid; place-items: center; font-family: var(--font-anthropic-serif); font-size: 1.05rem; color: var(--color-bg); background: var(--cds-chart-muted); }
.cx .mono.hi { background: var(--color-ok); } .cx .mono.lo { background: var(--color-bad); }
.cx .tile .nm { font-size: var(--cds-font-size-caption); font-weight: var(--cds-font-weight-medium); text-align: center; line-height: 1.2; overflow-wrap: anywhere; }
.cx .tile .st { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); font-variant-numeric: tabular-nums; }
.cx .detail { background: var(--color-panel); border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); padding: var(--cds-pad-lg); margin-bottom: var(--cds-gap-lg); scroll-margin-top: var(--cds-gap-lg); }
.cx .dhead { display: flex; align-items: center; gap: var(--cds-gap-md); margin-bottom: var(--cds-gap-md); flex-wrap: wrap; }
.cx .dhead .mono { width: 3.2rem; height: 3.2rem; font-size: 1.3rem; }
.cx .dhead h2 { font-family: var(--font-anthropic-serif); font-weight: 400; font-size: var(--cds-font-size-title); margin: 0; }
.cx .dhead .sub { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); }
.cx .dhead .close { margin-left: auto; font: inherit; font-size: var(--cds-font-size-caption); background: transparent; border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); color: var(--color-fg-muted); padding: var(--cds-pad-xs) var(--cds-pad-sm); cursor: pointer; }
.cx .kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 8.5rem), 1fr)); gap: var(--cds-gap-sm); margin-bottom: var(--cds-gap-lg); }
.cx .kpi { border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); padding: var(--cds-pad-sm) var(--cds-pad-md); background: var(--color-bg); }
.cx .kpi .lab { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); }
.cx .kpi .num { font-family: var(--font-anthropic-serif); font-size: var(--cds-font-size-heading); font-variant-numeric: tabular-nums; margin-top: var(--cds-gap-xs); }
.cx h3 { font-size: var(--cds-font-size-heading); font-weight: var(--cds-font-weight-medium); margin: var(--cds-gap-md) 0 var(--cds-gap-xs); }
.cx .note { font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); margin: 0 0 var(--cds-gap-sm); }
.cx .grid2 { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 22rem), 1fr)); gap: var(--cds-gap-lg); }
.cx .tblwrap { max-height: 22rem; overflow: auto; }
.cx table { width: 100%; border-collapse: collapse; font-size: var(--cds-font-size-caption); font-variant-numeric: tabular-nums; }
.cx th, .cx td { text-align: right; padding: var(--cds-pad-xs) var(--cds-pad-sm); border-bottom: 1px solid var(--color-border-line); white-space: nowrap; }
.cx th:first-child, .cx td:first-child { text-align: left; white-space: normal; }
.cx th { position: sticky; top: 0; background: var(--color-panel); color: var(--color-fg-muted); font-weight: var(--cds-font-weight-medium); }
.cx tr.pick { cursor: pointer; } .cx tr.pick:hover td { background: var(--color-bg); }
.cx .good { color: var(--color-ok); } .cx .bad { color: var(--color-bad); }
.cx .empty { color: var(--color-fg-muted); font-size: var(--cds-font-size-caption); padding: var(--cds-pad-sm) 0; }
.cx [hidden] { display: none !important; }
.cx td.edge { color: var(--color-warn); font-weight: var(--cds-font-weight-medium); }
.cx .edge-note { margin-top: calc(-1 * var(--cds-gap-sm)); font-size: var(--cds-font-size-caption); }
.cx [data-dash-source-state=refreshing] { opacity: .5; transition: opacity var(--cds-dur-slow); }
.cx .tl { border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); background: var(--color-bg); padding: var(--cds-pad-md); margin-bottom: var(--cds-gap-md); position: relative; }
.cx .tl-head { display: flex; flex-wrap: wrap; gap: var(--cds-gap-sm) var(--cds-gap-md); align-items: baseline; margin-bottom: var(--cds-gap-sm); }
.cx .tl-head b { font-weight: var(--cds-font-weight-medium); }
.cx .tl-head .r-win { color: var(--color-ok); } .cx .tl-head .r-loss { color: var(--color-bad); }
.cx .tl-nav { margin-left: auto; display: inline-flex; gap: var(--cds-gap-xs); }
.cx .tl-nav button { font: inherit; font-size: var(--cds-font-size-caption); background: transparent; border: 1px solid var(--color-border-line); border-radius: var(--cds-radius); color: var(--color-fg); padding: 0 var(--cds-pad-sm); cursor: pointer; }
.cx .tl-nav button:disabled { opacity: .4; cursor: default; }
.cx .tl-chart { min-height: 19rem; }
.cx .tl-legend { display: flex; flex-wrap: wrap; gap: var(--cds-gap-xs) var(--cds-gap-md); font-size: var(--cds-font-size-caption); color: var(--color-fg-muted); margin-top: var(--cds-gap-sm); align-items: center; }
.cx .tl-legend svg { vertical-align: middle; margin-right: .25rem; }
.cx tr.sel td { background: var(--color-bg); font-weight: var(--cds-font-weight-medium); }
.cx .tip { position: absolute; pointer-events: none; background: var(--cds-surface-popover); border: 1px solid var(--cds-border); border-radius: var(--cds-radius); box-shadow: var(--cds-shadow-popover); padding: var(--cds-pad-xs) var(--cds-pad-sm); font-size: var(--cds-font-size-caption); white-space: nowrap; z-index: 5; display: none; }
.cx .tip .t { color: var(--cds-text-secondary); }
@media (max-width: 34rem) { .cx .hide-narrow { display: none; } }
</style>

<div class="cx">
  <h1 id="top">{{PLAYER}} · Champions &amp; matchups</h1>
  <p class="take">Pick a champion to see everything about it from your <span data-source="overview" data-field="games">—</span> ranked solo games (<span data-source="overview" data-field="first_date">—</span> to <span data-source="overview" data-field="last_date">—</span>). <b>Your champions</b> shows how you play each one and who you've faced on it; <b>Enemies faced</b> shows how you do against each enemy laner. For bot games, enemies include both the enemy ADC and the enemy support. Tile color = win rate (3+ games).</p>
  <p class="take edge-note" data-source="favored">★ marks the champion favored in a lane matchup, ★★ a hard counter; no star means no clear edge (or too few games to judge). Ratings come from lolalytics.com: <span data-source="favored_info" data-field="summary">—</span> For bot they compare same roles only (ADC vs ADC, support vs support).</p>

  <div class="bar">
    <div class="seg big" id="side"><button type="button" data-v="mine" aria-pressed="true">Your champions</button><button type="button" data-v="enemy" aria-pressed="false">Enemies faced</button></div>
    <div class="seg" id="role"><button type="button" data-v="All" aria-pressed="true">All roles</button><button type="button" data-v="Top" aria-pressed="false">Top</button><button type="button" data-v="Bot" aria-pressed="false">Bot</button><button type="button" data-v="Jungle" aria-pressed="false">Jungle</button></div>
    <input type="search" id="q" placeholder="Find a champion" aria-label="Find a champion">
  </div>

  <section class="detail" id="detail" hidden>
    <div class="dhead"><div class="mono" id="d-mono"></div><div><h2 id="d-name"></h2><div class="sub" id="d-sub"></div></div><button type="button" class="close" id="d-close">Close ×</button></div>

    <div id="d-mine" hidden>
      <div class="kpis">
        <div class="kpi"><div class="lab">Record</div><div class="num" data-source="mine" data-field="record">—</div></div>
        <div class="kpi"><div class="lab">Win rate</div><div class="num" data-source="mine" data-field="wr_txt">—</div></div>
        <div class="kpi"><div class="lab">K / D / A</div><div class="num" data-source="mine" data-field="kda">—</div></div>
        <div class="kpi"><div class="lab">CS / min</div><div class="num" data-source="mine" data-field="cspm_txt">—</div></div>
        <div class="kpi"><div class="lab">Kill participation</div><div class="num" data-source="mine" data-field="kp_txt">—</div></div>
        <div class="kpi"><div class="lab">Damage share</div><div class="num" data-source="mine" data-field="dmg_txt">—</div></div>
        <div class="kpi"><div class="lab">Lane gold @10</div><div class="num" data-source="mine" data-field="gold10_txt">—</div></div>
        <div class="kpi"><div class="lab">Lane gold @15</div><div class="num" data-source="mine" data-field="gold15_txt">—</div></div>
        <div class="kpi"><div class="lab">Your deaths before 14</div><div class="num" data-source="mine" data-field="early_txt">—</div></div>
        <div class="kpi"><div class="lab">Avg game</div><div class="num" data-source="mine" data-field="minutes_txt">—</div></div>
      </div>
      <p class="note">Lane gold = your gold minus your lane opponent's; in bot games it's your duo minus theirs. Click an opponent to open them.</p>
      <div class="grid2" id="d-mine-tables"></div>
    </div>

    <div id="d-enemy" hidden>
      <div class="kpis">
        <div class="kpi"><div class="lab">Your record vs them</div><div class="num" data-source="enemies" data-field="record">—</div></div>
        <div class="kpi"><div class="lab">Win rate</div><div class="num" data-source="enemies" data-field="wr_txt">—</div></div>
        <div class="kpi"><div class="lab">Lane gold @10</div><div class="num" data-source="enemies" data-field="gold10_txt">—</div></div>
        <div class="kpi"><div class="lab">Lane gold @15</div><div class="num" data-source="enemies" data-field="gold15_txt">—</div></div>
        <div class="kpi"><div class="lab">Your deaths before 14</div><div class="num" data-source="enemies" data-field="early_txt">—</div></div>
        <div class="kpi"><div class="lab">Your K / D / A</div><div class="num" data-source="enemies" data-field="kda">—</div></div>
      </div>
      <h3 id="d-enemy-h">What you played against them</h3>
      <p class="note">Click one of your champions to open it.</p>
      <div class="tblwrap"><table id="t-enemy-mine" data-source="enemy_by_mine" data-row-key="key">
        <thead><tr><th data-field="mine">Your champion</th><th data-field="edge" data-source="favored">Lane edge</th><th data-field="as">As</th><th data-field="games">Games</th><th data-field="record">W–L</th><th data-field="wr_txt">Win rate</th><th data-field="gold10_txt">Gold @10</th><th data-field="gold15_txt">Gold @15</th></tr></thead><tbody></tbody></table></div>
    </div>

    <h3 id="d-timeline-h">Game timeline</h3>
    <p class="note" id="d-timeline-note">Gold vs your lane opponent minute by minute, with your kills, deaths and assists on the line and every objective underneath. Click a game in the list below to switch.</p>
    <div class="tl" id="tl">
      <div class="tl-head"><span id="tl-title"></span><span class="tl-nav"><button type="button" id="tl-prev" aria-label="Previous game">← Older</button><button type="button" id="tl-next" aria-label="Next game">Newer →</button></span></div>
      <div class="tl-chart" id="tl-chart" data-source="game_gold game_events"></div>
      <div class="tl-legend" id="tl-legend"></div>
    </div>
    <h3 id="d-games-h">Games</h3>
    <div class="tblwrap"><table id="t-games" data-source="game_rows" data-row-key="match_id">
      <thead><tr><th data-field="date">Date</th><th data-field="result">Result</th><th data-field="champion">You</th><th data-field="role" class="hide-narrow">Role</th><th data-field="vs">Against</th><th data-field="kda">K / D / A</th><th data-field="gold15_txt">Gold @15</th></tr></thead><tbody></tbody></table></div>
  </section>

  <div class="tiles" id="tiles" data-source="mine enemies"></div>
  <p class="empty" id="tiles-empty" hidden>No champions match.</p>
</div>

<script>
dash.calc("favored_info", {
  title: "Matchup ratings source", inputs: ["favored"],
  description: "Time window and count of matchup ratings used for the ★ marks.",
  fn: (favored) => {
    const patches = [...new Set(favored.map((r) => r.patch))].sort();
    return [{ pairs: favored.length, summary: favored.length.toLocaleString() + " matchups across every champion in each role, " + (patches.every((p) => /^\d+d$/.test(p)) ? "last " + parseInt(patches[0]) + " days" : "patch " + (patches.length > 1 ? patches[0] + "–" + patches[patches.length - 1] : patches[0])) + ", Emerald+, using its opponent-adjusted win-rate edge (★ at 2+ points, ★★ at 4+, matchups with 200+ games)." }];
  },
});

dash.calc("overview", {
  title: "Games covered", inputs: ["champ_games"], description: "Number of games and date range.",
  fn: (champ_games) => { const d = champ_games.map((g) => g.date).sort(); return [{ games: champ_games.length, first_date: d[0], last_date: d[d.length - 1] }]; },
});

dash.calc("mine", {
  title: "Your champions", inputs: ["champ_games"],
  description: "Per champion you played: record, averages, and gold vs your lane opponent (duo vs duo in bot) at 10 and 15 minutes.",
  fn: (champ_games) => {
    const num = (v) => (v === "" || v == null ? null : Number(v));
    const avg = (xs, f) => { const v = xs.map(f).filter((n) => n != null && !isNaN(n)); return v.length ? v.reduce((a, b) => a + b, 0) / v.length : null; };
    const sgn = (n) => (n == null ? "—" : (n >= 0 ? "+" : "−") + Math.abs(Math.round(n)).toLocaleString("en-US"));
    const isBot = (g) => g.role === "ADC" || g.role === "Support";
    const by = {}; champ_games.forEach((g) => (by[g.champion] = by[g.champion] || []).push(g));
    return Object.entries(by).map(([champ, xs]) => {
      const w = xs.filter((g) => g.win === "1").length, roles = {};
      xs.forEach((g) => { roles[g.role] = (roles[g.role] || 0) + 1; });
      const g10 = avg(xs, (g) => num(isBot(g) ? g.duo_gold_10 : g.lane_gold_10)), g15 = avg(xs, (g) => num(isBot(g) ? g.duo_gold_15 : g.lane_gold_15));
      const k = avg(xs, (g) => num(g.kills)), d = avg(xs, (g) => num(g.deaths)), a = avg(xs, (g) => num(g.assists));
      return {
        champ, games: xs.length, wins: w, wr: w / xs.length, record: w + "–" + (xs.length - w), wr_txt: Math.round((100 * w) / xs.length) + "%",
        roles: Object.entries(roles).sort((p, q) => q[1] - p[1]).map(([r, n]) => r + " " + n).join(" · "),
        bot_games: xs.filter(isBot).length, top_games: roles.Top || 0, jg_games: roles.Jungle || 0,
        kda: k.toFixed(1) + " / " + d.toFixed(1) + " / " + a.toFixed(1),
        cspm_txt: avg(xs, (g) => num(g.cs_per_min)).toFixed(1), kp_txt: Math.round(100 * avg(xs, (g) => num(g.kill_participation))) + "%",
        dmg_txt: Math.round(100 * avg(xs, (g) => num(g.damage_share))) + "%",
        gold10_txt: sgn(g10), gold15_txt: sgn(g15), early_txt: avg(xs, (g) => num(g.early_deaths)).toFixed(1),
        minutes_txt: Math.round(avg(xs, (g) => num(g.minutes))) + " min",
      };
    }).sort((p, q) => q.games - p.games || q.wr - p.wr);
  },
});

dash.calc("mine_vs", {
  title: "Your champions vs opponents", inputs: ["champ_games"],
  description: "For each champion you played: record and gold diff against each lane opponent (enemy ADC and enemy support separately in bot), and with each lane partner.",
  fn: (champ_games) => {
    const num = (v) => (v === "" || v == null ? null : Number(v));
    const avg = (xs, f) => { const v = xs.map(f).filter((n) => n != null && !isNaN(n)); return v.length ? v.reduce((a, b) => a + b, 0) / v.length : null; };
    const sgn = (n) => (n == null ? "—" : (n >= 0 ? "+" : "−") + Math.abs(Math.round(n)).toLocaleString("en-US"));
    const groups = {};
    champ_games.forEach((g) => {
      const bot = g.role === "ADC" || g.role === "Support";
      const add = (type, vs) => { if (!vs) return; const key = g.champion + "|" + type + "|" + vs; (groups[key] = groups[key] || { champ: g.champion, type, vs, xs: [], bot }).xs.push(g); };
      if (bot) { add("Enemy ADC", g.enemy_adc); add("Enemy support", g.enemy_support); add("Lane partner", g.partner); }
      else add("Lane opponent", g.opponent);
    });
    return Object.entries(groups).map(([key, o]) => {
      const w = o.xs.filter((g) => g.win === "1").length;
      return { key, champ: o.champ, type: o.type, vs: o.vs, games: o.xs.length, wr: w / o.xs.length, record: w + "–" + (o.xs.length - w),
        wr_txt: Math.round((100 * w) / o.xs.length) + "%",
        gold10_txt: sgn(avg(o.xs, (g) => num(o.bot ? g.duo_gold_10 : g.lane_gold_10))), gold15_txt: sgn(avg(o.xs, (g) => num(o.bot ? g.duo_gold_15 : g.lane_gold_15))) };
    }).sort((p, q) => q.games - p.games || q.wr - p.wr);
  },
});

dash.calc("enemies", {
  title: "Enemies faced", inputs: ["champ_games"],
  description: "Per enemy champion you laned against (your direct opponent, or the enemy ADC or support in bot games): your record, gold diff and early deaths.",
  fn: (champ_games) => {
    const num = (v) => (v === "" || v == null ? null : Number(v));
    const avg = (xs, f) => { const v = xs.map(f).filter((n) => n != null && !isNaN(n)); return v.length ? v.reduce((a, b) => a + b, 0) / v.length : null; };
    const sgn = (n) => (n == null ? "—" : (n >= 0 ? "+" : "−") + Math.abs(Math.round(n)).toLocaleString("en-US"));
    const by = {};
    champ_games.forEach((g) => {
      const bot = g.role === "ADC" || g.role === "Support";
      const list = bot ? [[g.enemy_adc, "Bot (ADC)"], [g.enemy_support, "Bot (support)"]] : [[g.opponent, g.role]];
      list.forEach(([e, where]) => { if (!e) return; (by[e] = by[e] || { xs: [], where: {} }).xs.push({ g, bot }); by[e].where[where] = (by[e].where[where] || 0) + 1; });
    });
    return Object.entries(by).map(([champ, o]) => {
      const xs = o.xs, w = xs.filter(({ g }) => g.win === "1").length;
      const k = avg(xs, ({ g }) => num(g.kills)), d = avg(xs, ({ g }) => num(g.deaths)), a = avg(xs, ({ g }) => num(g.assists));
      return { champ, games: xs.length, wins: w, wr: w / xs.length, record: w + "–" + (xs.length - w), wr_txt: Math.round((100 * w) / xs.length) + "%",
        faced_as: Object.entries(o.where).sort((p, q) => q[1] - p[1]).map(([r, n]) => r + " " + n).join(" · "),
        top_games: o.where.Top || 0, jg_games: o.where.Jungle || 0, bot_games: (o.where["Bot (ADC)"] || 0) + (o.where["Bot (support)"] || 0),
        gold10_txt: sgn(avg(xs, ({ g, bot }) => num(bot ? g.duo_gold_10 : g.lane_gold_10))), gold15_txt: sgn(avg(xs, ({ g, bot }) => num(bot ? g.duo_gold_15 : g.lane_gold_15))),
        early_txt: avg(xs, ({ g }) => num(g.early_deaths)).toFixed(1), kda: k.toFixed(1) + " / " + d.toFixed(1) + " / " + a.toFixed(1) };
    }).sort((p, q) => q.games - p.games || q.wr - p.wr);
  },
});

dash.calc("enemy_by_mine", {
  title: "Enemies by your champion", inputs: ["champ_games"],
  description: "For each enemy laner: which of your champions you played against them, with record and gold diff.",
  fn: (champ_games) => {
    const num = (v) => (v === "" || v == null ? null : Number(v));
    const avg = (xs, f) => { const v = xs.map(f).filter((n) => n != null && !isNaN(n)); return v.length ? v.reduce((a, b) => a + b, 0) / v.length : null; };
    const sgn = (n) => (n == null ? "—" : (n >= 0 ? "+" : "−") + Math.abs(Math.round(n)).toLocaleString("en-US"));
    const groups = {};
    champ_games.forEach((g) => {
      const bot = g.role === "ADC" || g.role === "Support";
      const list = bot ? [g.enemy_adc, g.enemy_support] : [g.opponent];
      list.forEach((e) => { if (!e) return; const key = e + "|" + g.champion + "|" + g.role; (groups[key] = groups[key] || { enemy: e, mine: g.champion, as: g.role, bot, xs: [] }).xs.push(g); });
    });
    return Object.entries(groups).map(([key, o]) => {
      const w = o.xs.filter((g) => g.win === "1").length;
      return { key, enemy: o.enemy, mine: o.mine, as: o.as, games: o.xs.length, record: w + "–" + (o.xs.length - w), wr: w / o.xs.length,
        wr_txt: Math.round((100 * w) / o.xs.length) + "%",
        gold10_txt: sgn(avg(o.xs, (g) => num(o.bot ? g.duo_gold_10 : g.lane_gold_10))), gold15_txt: sgn(avg(o.xs, (g) => num(o.bot ? g.duo_gold_15 : g.lane_gold_15))) };
    }).sort((p, q) => q.games - p.games || q.wr - p.wr);
  },
});

dash.calc("game_rows", {
  title: "Game list", inputs: ["champ_games"],
  description: "Each game with result, who you laned against (enemy duo in bot), your K/D/A and gold diff at 15.",
  fn: (champ_games) => champ_games.map((g) => {
    const bot = g.role === "ADC" || g.role === "Support";
    const n = bot ? g.duo_gold_15 : g.lane_gold_15;
    return { match_id: g.match_id, date: g.date, result: g.win === "1" ? "Win" : "Loss", champion: g.champion, role: g.role,
      vs: bot ? [g.enemy_adc, g.enemy_support].filter(Boolean).join(" + ") : g.opponent,
      enemies: (bot ? [g.enemy_adc, g.enemy_support] : [g.opponent]).filter(Boolean).join("|"),
      kda: g.kills + " / " + g.deaths + " / " + g.assists,
      gold15_txt: n === "" || n == null ? "—" : (Number(n) >= 0 ? "+" : "−") + Math.abs(Number(n)).toLocaleString("en-US") };
  }).reverse(),
});
</script>

<script>
(() => {
  const $ = (s) => document.querySelector(s);
  const P = () => ({ side: "mine", role: "All", champ: null, game: null, ...dash.params() });
  const enc = (s) => encodeURIComponent(s);
  const initials = (name) => { const w = name.replace(/[^A-Za-z' .&]/g, "").split(/[\s.&']+/).filter(Boolean); return (w.length > 1 ? w[0][0] + w[1][0] : name.slice(0, 2)).toUpperCase(); };
  const tone = (r) => (r.games >= 3 ? (r.wr >= 0.55 ? "hi" : r.wr <= 0.4 ? "lo" : "") : "");
  const cls = (txt) => (txt.startsWith("+") ? "good" : txt.startsWith("−") ? "bad" : "");

  const seg = (id, key) => document.querySelectorAll(`#${id} button`).forEach((b) => b.addEventListener("click", () => {
    const upd = { [key]: b.dataset.v }; if (key === "side") { upd.champ = null; upd.game = null; } dash.setParams(upd);
  }));
  seg("side", "side"); seg("role", "role");
  $("#q").addEventListener("input", () => draw());
  $("#d-close").addEventListener("click", () => dash.setParams({ champ: null, game: null }));
  const open = (side, champ) => { dash.setParams({ side, champ, game: null }); requestAnimationFrame(() => $("#detail").scrollIntoView({ block: "start" })); };

  function roleOk(r, role) {
    if (role === "All") return true;
    if (role === "Top") return r.top_games > 0;
    if (role === "Jungle") return r.jg_games > 0;
    return r.bot_games > 0;
  }

  function tiles(p) {
    const box = $("#tiles"); box.replaceChildren();
    const src = dash.data(p.side === "mine" ? "mine" : "enemies");
    if (src.status !== "ok") return;
    const q = $("#q").value.trim().toLowerCase();
    const rows = src.data.filter((r) => roleOk(r, p.role) && (!q || r.champ.toLowerCase().includes(q)));
    $("#tiles-empty").hidden = rows.length > 0;
    rows.forEach((r) => {
      const b = document.createElement("button"); b.type = "button"; b.className = "tile"; b.setAttribute("aria-pressed", String(r.champ === p.champ));
      const m = document.createElement("div"); m.className = "mono " + tone(r); m.textContent = initials(r.champ);
      const n = document.createElement("div"); n.className = "nm"; n.textContent = r.champ;
      const s = document.createElement("div"); s.className = "st"; s.textContent = r.games + (r.games === 1 ? " game" : " games") + " · " + r.wr_txt;
      b.append(m, n, s); b.addEventListener("click", () => open(p.side, r.champ)); box.appendChild(b);
    });
  }

  function fillKpis(scope, src, champ) {
    scope.querySelectorAll(`[data-source="${src}"]`).forEach((el) => {
      const d = dash.data(src), r = d.status === "ok" ? d.data.find((x) => x.champ === champ) : null;
      el.setAttribute("data-where", "champ=" + enc(champ));
      el.textContent = r ? String(r[el.dataset.field]) : "—";
      el.classList.remove("good", "bad"); if (/gold/.test(el.dataset.field) && r) { const c = cls(String(r[el.dataset.field])); if (c) el.classList.add(c); }
    });
  }

  function table(rows, cols, onPick, src, keyField) {
    const wrap = document.createElement("div"); wrap.className = "tblwrap";
    const t = document.createElement("table"); t.setAttribute("data-source", src); t.setAttribute("data-row-key", keyField);
    const hr = document.createElement("tr");
    cols.forEach(([f, lab, fn]) => { const th = document.createElement("th"); th.setAttribute("data-field", fn ? "edge" : f); if (fn) th.setAttribute("data-source", "favored"); th.textContent = lab; hr.appendChild(th); });
    const th = document.createElement("thead"); th.appendChild(hr); t.appendChild(th);
    const tb = document.createElement("tbody");
    rows.forEach((r) => {
      const tr = document.createElement("tr"); tr.setAttribute("data-key", r[keyField]);
      if (onPick) { tr.className = "pick"; tr.addEventListener("click", () => onPick(r)); }
      cols.forEach(([f, , fn]) => { const td = document.createElement("td"); td.textContent = fn ? fn(r) : String(r[f]); if (/gold/.test(f)) td.className = cls(String(r[f])); if (fn) td.className = "edge"; tr.appendChild(td); });
      tb.appendChild(tr);
    });
    t.appendChild(tb); wrap.appendChild(t); return wrap;
  }

  function detail(p) {
    const box = $("#detail");
    if (!p.champ) { box.hidden = true; return; }
    const src = dash.data(p.side === "mine" ? "mine" : "enemies");
    const r = src.status === "ok" ? src.data.find((x) => x.champ === p.champ) : null;
    if (src.status === "ok" && !r) { box.hidden = true; return; }
    box.hidden = false;
    $("#d-name").textContent = p.champ;
    $("#d-mono").textContent = initials(p.champ); $("#d-mono").className = "mono " + (r ? tone(r) : "");
    $("#d-sub").textContent = r ? (p.side === "mine" ? "You played: " + r.roles : "You faced them: " + r.faced_as) : "";
    $("#d-mine").hidden = p.side !== "mine"; $("#d-enemy").hidden = p.side !== "enemy";

    if (p.side === "mine") {
      fillKpis($("#d-mine"), "mine", p.champ);
      const box2 = $("#d-mine-tables"); box2.replaceChildren();
      const vs = dash.data("mine_vs");
      if (vs.status === "ok") {
        const mineRows = vs.data.filter((x) => x.champ === p.champ);
        ["Lane opponent", "Enemy ADC", "Enemy support", "Lane partner"].forEach((type) => {
          const rows = mineRows.filter((x) => x.type === type); if (!rows.length) return;
          const card = document.createElement("div");
          const h = document.createElement("h3"); h.textContent = type === "Lane opponent" ? "Against lane opponents" : type === "Lane partner" ? "With your lane partner" : "Against the " + type.toLowerCase();
          card.appendChild(h);
          const pick = type === "Lane partner" ? null : (x) => open("enemy", x.vs);
          const roleFor = type === "Enemy ADC" ? ["ADC"] : type === "Enemy support" ? ["Support"] : ["Top", "Jungle", "Mid"];
          const edgeCol = ["edge", "Lane edge", (x) => edgeLabel(roleFor, p.champ, x.vs)];
          const cols = [["vs", type === "Lane partner" ? "Partner" : "Champion"], ["games", "Games"], ["record", "W–L"], ["wr_txt", "Win rate"], ["gold10_txt", "Gold @10"], ["gold15_txt", "Gold @15"]];
          if (type !== "Lane partner") cols.splice(1, 0, edgeCol);
          card.appendChild(table(rows, cols, pick, "mine_vs", "key"));
          box2.appendChild(card);
        });
      }
    } else {
      fillKpis($("#d-enemy"), "enemies", p.champ);
      $("#d-enemy-h").textContent = "What you played against " + p.champ;
      const t = $("#t-enemy-mine tbody"); t.replaceChildren();
      const em = dash.data("enemy_by_mine");
      if (em.status === "ok") em.data.filter((x) => x.enemy === p.champ).forEach((x) => {
        const tr = document.createElement("tr"); tr.className = "pick"; tr.setAttribute("data-key", x.key); tr.addEventListener("click", () => open("mine", x.mine));
        [x.mine, edgeLabel([x.as], x.mine, p.champ), x.as, x.games, x.record, x.wr_txt, x.gold10_txt, x.gold15_txt].forEach((v, i) => { const td = document.createElement("td"); td.textContent = String(v); if (i >= 6) td.className = cls(String(v)); if (i === 1) td.className = "edge"; tr.appendChild(td); });
        t.appendChild(tr);
      });
    }

    $("#d-games-h").textContent = p.side === "mine" ? "Your " + p.champ + " games" : "Games against " + p.champ;
    const gt = $("#t-games tbody"); gt.replaceChildren();
    const gr = dash.data("game_rows");
    const list = gr.status === "ok" ? gr.data.filter((g) => (p.side === "mine" ? g.champion === p.champ : g.enemies.split("|").includes(p.champ))) : [];
    const selGame = list.find((g) => g.match_id === p.game) || list[0];
    timeline(selGame, list, p);
    list.forEach((g) => {
      const tr = document.createElement("tr"); tr.setAttribute("data-key", g.match_id); tr.className = "pick" + (selGame && g.match_id === selGame.match_id ? " sel" : "");
      tr.addEventListener("click", () => { dash.setParams({ game: g.match_id }); requestAnimationFrame(() => $("#tl").scrollIntoView({ block: "nearest" })); });
      const gx = gameEdge(g);
      [g.date, g.result, gx.mine, g.role, gx.vs, g.kda, g.gold15_txt].forEach((v, i) => {
        const td = document.createElement("td"); td.textContent = String(v);
        if (i === 1) td.className = v === "Win" ? "good" : "bad"; if (i === 3) td.className = "hide-narrow"; if (i === 6) td.className = cls(String(v));
        tr.appendChild(td);
      });
      gt.appendChild(tr);
    });
  }


  const tip = document.createElement("div"); tip.className = "tip"; document.querySelector(".cx").appendChild(tip);
  const root = document.querySelector(".cx");
  const showTip = (ev, lines) => { tip.replaceChildren(); lines.forEach(([c, t]) => { const d = document.createElement("div"); if (c) d.className = c; d.textContent = t; tip.appendChild(d); });
    tip.style.display = "block"; const rb = root.getBoundingClientRect(), tw = tip.offsetWidth;
    let x = ev.clientX - rb.left + 12; if (x + tw > rb.width) x = ev.clientX - rb.left - tw - 12;
    tip.style.left = Math.max(0, x) + "px"; tip.style.top = (ev.clientY - rb.top + 12) + "px"; };
  const hideTip = () => { tip.style.display = "none"; };
  const clock = (m) => { const s = Math.round(m * 60); return Math.floor(s / 60) + ":" + String(s % 60).padStart(2, "0"); };
  const fmtG = (n) => (n >= 0 ? "+" : "−") + d3.format(",")(Math.abs(Math.round(n)));
  const OBJ = { Dragon: "D", "Elder dragon": "E", Herald: "H", Grubs: "G", Baron: "B", Tower: "T", Inhibitor: "I" };

  function glyph(sel, kind, size) {
    const s = size;
    if (kind === "Kill") return sel.append("path").attr("d", `M0,${-s} L${s * 0.9},${s * 0.6} L${-s * 0.9},${s * 0.6} Z`).attr("fill", "var(--color-ok)").attr("stroke", "var(--color-bg)").attr("stroke-width", 1.5);
    if (kind === "Death") { const g = sel.append("g"); g.append("circle").attr("r", s * 0.95).attr("fill", "var(--color-bad)").attr("stroke", "var(--color-bg)").attr("stroke-width", 1.5);
      g.append("path").attr("d", `M${-s * 0.4},${-s * 0.4} L${s * 0.4},${s * 0.4} M${s * 0.4},${-s * 0.4} L${-s * 0.4},${s * 0.4}`).attr("stroke", "var(--color-bg)").attr("stroke-width", 1.8).attr("stroke-linecap", "round"); return g; }
    if (kind === "Assist") return sel.append("circle").attr("r", s * 0.55).attr("fill", dash.colors[0]).attr("stroke", "var(--color-bg)").attr("stroke-width", 1.5);
  }
  function badge(sel, letter, ours, r) {
    const g = sel.append("g");
    g.append("circle").attr("r", r).attr("fill", ours ? "var(--color-ok)" : "var(--color-bad)").attr("stroke", "var(--color-bg)").attr("stroke-width", 1.5);
    g.append("text").attr("text-anchor", "middle").attr("dy", "0.35em").attr("font-size", r * 1.15).attr("font-weight", 600).attr("fill", "var(--color-bg)").text(letter);
    return g;
  }

  const legend = $("#tl-legend");
  (() => {
    const add = (draw, label) => { const sp = document.createElement("span"); const svg = d3.select(sp).append("svg").attr("width", 16).attr("height", 16).append("g").attr("transform", "translate(8,8)"); draw(svg); sp.appendChild(document.createTextNode(label)); legend.appendChild(sp); };
    add((g) => glyph(g, "Kill", 6), "Your kill"); add((g) => glyph(g, "Death", 6), "Your death"); add((g) => glyph(g, "Assist", 6), "Your assist");
    add((g) => badge(g, "D", true, 7), "Dragon (E = Elder)"); add((g) => badge(g, "G", true, 7), "Grubs"); add((g) => badge(g, "H", true, 7), "Herald");
    add((g) => badge(g, "B", true, 7), "Baron"); add((g) => badge(g, "T", true, 7), "Tower"); add((g) => badge(g, "I", true, 7), "Inhibitor");
    const sp = document.createElement("span"); sp.textContent = "Green = your team · red = enemy"; legend.appendChild(sp);
  })();

  function timeline(g, list, p) {
    const box = $("#tl-chart"); box.replaceChildren(); hideTip();
    const idx = g ? list.indexOf(g) : -1;
    $("#tl-prev").disabled = !g || idx >= list.length - 1; $("#tl-next").disabled = !g || idx <= 0;
    $("#tl-prev").onclick = () => dash.setParams({ game: list[idx + 1].match_id });
    $("#tl-next").onclick = () => dash.setParams({ game: list[idx - 1].match_id });
    const title = $("#tl-title"); title.replaceChildren();
    if (!g) return;
    const res = document.createElement("b"); res.className = g.result === "Win" ? "r-win" : "r-loss"; res.textContent = g.result;
    const gx = gameEdge(g);
    title.append(res, document.createTextNode(" · " + g.date + " · " + gx.mine + " (" + g.role + ") vs " + gx.vs + " · " + g.kda));
    const bot = g.role === "ADC" || g.role === "Support";
    $("#d-timeline-note").textContent = (bot ? "Your duo's gold vs the enemy duo" : "Your gold vs your lane opponent") + " minute by minute (solid line), your team vs theirs (dashed), with your kills, deaths and assists on the line and every objective underneath. Click a game in the list below to switch.";
    const gg = dash.data("game_gold"), ge = dash.data("game_events");
    if (gg.status === "error" || ge.status === "error") { const d = document.createElement("div"); d.className = "empty"; d.textContent = "Failed to load"; box.appendChild(d); return; }
    if (gg.status !== "ok" || ge.status !== "ok") return;
    const pts = gg.data.filter((r) => r.match_id === g.match_id).map((r) => ({ m: Number(r.minute), v: r.gold_diff === "" ? null : Number(r.gold_diff), t: Number(r.team_gold_diff) })).sort((a, b) => a.m - b.m);
    const evs = ge.data.filter((r) => r.match_id === g.match_id).map((r) => ({ ...r, m: Number(r.minute) }));
    if (!pts.length) { const d = document.createElement("div"); d.className = "empty"; d.textContent = "No timeline for this game."; box.appendChild(d); return; }

    const W = box.clientWidth || 600, plotH = 210, stripH = 64, H = plotH + stripH + 26;
    const svg = d3.select(box).append("svg").attr("width", "100%").attr("viewBox", `0 0 ${W} ${H}`);
    const line = pts.filter((q) => q.v != null);
    const ext = d3.extent(line.flatMap((q) => [q.v, q.t]).concat([0]));
    const pad = Math.max(500, (ext[1] - ext[0]) * 0.08);
    const y = d3.scaleLinear([ext[0] - pad, ext[1] + pad], [plotH - 4, 10]).nice();
    const tw = (s) => { const t = svg.append("text").attr("font-size", "var(--cds-font-size-caption)").text(s); const w = t.node().getComputedTextLength(); t.remove(); return w; };
    const ml = d3.max(y.ticks(5), (t) => tw(t === 0 ? "0" : fmtG(t))) + 8, mr = 10;
    const maxM = Math.max(d3.max(pts, (q) => q.m), d3.max(evs, (e) => e.m) || 0);
    const x = d3.scaleLinear([0, maxM], [ml, W - mr]);

    y.ticks(5).forEach((t) => {
      svg.append("line").attr("x1", ml).attr("x2", W - mr).attr("y1", y(t)).attr("y2", y(t)).attr("stroke", t === 0 ? "var(--cds-chart-axis)" : "var(--cds-chart-grid)");
      svg.append("text").attr("x", ml - 6).attr("y", y(t)).attr("dy", "0.35em").attr("text-anchor", "end").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg-muted)").text(t === 0 ? "0" : fmtG(t));
    });
    const xt = x.ticks(Math.max(3, Math.min(10, Math.floor((W - ml) / 60))));
    xt.forEach((t) => {
      svg.append("line").attr("x1", x(t)).attr("x2", x(t)).attr("y1", 10).attr("y2", plotH + stripH).attr("stroke", "var(--cds-chart-grid)").attr("stroke-dasharray", "2 4");
      svg.append("text").attr("x", x(t)).attr("y", H - 6).attr("text-anchor", "middle").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg-muted)").text(t + "m");
    });

    // ahead/behind fill
    const uid = "tl" + Math.random().toString(36).slice(2, 8);
    const defs = svg.append("defs");
    defs.append("clipPath").attr("id", uid + "a").append("rect").attr("x", 0).attr("y", 0).attr("width", W).attr("height", y(0));
    defs.append("clipPath").attr("id", uid + "b").append("rect").attr("x", 0).attr("y", y(0)).attr("width", W).attr("height", plotH - y(0));
    const area = d3.area((q) => x(q.m), y(0), (q) => y(q.v));
    svg.append("path").datum(line).attr("d", area).attr("fill", "var(--color-ok)").attr("opacity", 0.14).attr("clip-path", `url(#${uid}a)`);
    svg.append("path").datum(line).attr("d", area).attr("fill", "var(--color-bad)").attr("opacity", 0.14).attr("clip-path", `url(#${uid}b)`);
    svg.append("path").datum(pts).attr("d", d3.line((q) => x(q.m), (q) => y(q.t))).attr("fill", "none").attr("stroke", "var(--cds-chart-muted)").attr("stroke-width", 1.5).attr("stroke-dasharray", "5 4");
    svg.append("path").datum(line).attr("d", d3.line((q) => x(q.m), (q) => y(q.v))).attr("fill", "none").attr("stroke", "var(--color-fg)").attr("stroke-width", 2).attr("stroke-linejoin", "round").attr("stroke-linecap", "round");
    svg.append("text").attr("x", ml + 4).attr("y", 14).attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg-muted)").text(bot ? "Ahead of enemy duo" : "Ahead of opponent");
    svg.append("text").attr("x", ml + 4).attr("y", plotH - 8).attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg-muted)").text("Behind");

    const valAt = (m) => { const i = d3.bisector((q) => q.m).left(line, m); if (i <= 0) return line[0].v; if (i >= line.length) return line[line.length - 1].v;
      const a = line[i - 1], b = line[i]; return a.v + (b.v - a.v) * ((m - a.m) / (b.m - a.m)); };

    // hover rule for gold values
    const rule = svg.append("line").attr("y1", 10).attr("y2", plotH).attr("stroke", "var(--cds-chart-reference)").style("display", "none");
    svg.append("rect").attr("x", ml).attr("y", 0).attr("width", W - ml - mr).attr("height", plotH).attr("fill", "transparent")
      .on("mousemove", (ev) => { const mn = Math.round(x.invert(d3.pointer(ev)[0])); const q = pts.find((z) => z.m === mn); if (!q) return hideTip();
        rule.style("display", null).attr("x1", x(mn)).attr("x2", x(mn));
        showTip(ev, [["t", "Minute " + mn], [null, (bot ? "Duo gold: " : "Lane gold: ") + (q.v == null ? "—" : fmtG(q.v))], [null, "Team gold: " + fmtG(q.t)]]); })
      .on("mouseleave", () => { rule.style("display", "none"); hideTip(); });

    // personal events on the line (assists first so kills/deaths draw on top)
    const order = { Assist: 0, Kill: 1, Death: 2 };
    evs.filter((e) => e.side === "You").sort((a, b) => order[a.kind] - order[b.kind]).forEach((e) => {
      const gEl = svg.append("g").attr("transform", `translate(${x(e.m)},${y(valAt(e.m))})`).style("cursor", "default");
      glyph(gEl, e.kind, e.kind === "Assist" ? 6 : 7);
      gEl.append("circle").attr("r", 9).attr("fill", "transparent")
        .on("mousemove", (ev) => showTip(ev, [["t", clock(e.m) + " · " + e.kind], [null, e.detail]])).on("mouseleave", hideTip);
    });

    // objective strip: two rows
    const rows = [["Your team", plotH + 18], ["Enemy team", plotH + 46]];
    rows.forEach(([lab, yy]) => {
      svg.append("text").attr("x", ml - 6).attr("y", yy).attr("dy", "0.35em").attr("text-anchor", "end").attr("font-size", "var(--cds-font-size-caption)").attr("fill", "var(--color-fg-muted)").text(lab === "Your team" ? "Yours" : "Theirs");
      svg.append("line").attr("x1", ml).attr("x2", W - mr).attr("y1", yy).attr("y2", yy).attr("stroke", "var(--cds-chart-grid)");
    });
    const objs = evs.filter((e) => e.side !== "You" && OBJ[e.kind]);
    const lastX = {};
    objs.forEach((e) => {
      const ours = e.side === "Your team", yy = ours ? rows[0][1] : rows[1][1];
      let cx = x(e.m); const key = ours ? "a" : "b"; let dy = 0;
      if (lastX[key] != null && cx - lastX[key].x < 13) { dy = lastX[key].dy === 0 ? -9 : 0; } lastX[key] = { x: cx, dy };
      const gEl = svg.append("g").attr("transform", `translate(${cx},${yy + dy})`);
      badge(gEl, OBJ[e.kind], ours, e.kind === "Tower" || e.kind === "Grubs" ? 6.5 : 8);
      gEl.append("circle").attr("r", 9).attr("fill", "transparent")
        .on("mousemove", (ev) => showTip(ev, [["t", clock(e.m) + " · " + (ours ? "Your team" : "Enemy team")], [null, e.detail]])).on("mouseleave", hideTip);
    });
  }


  // ---- favored-matchup marks (thresholds: edge in win-rate points, min games) ----
  const EDGE_FAVORED = 2, EDGE_HARD = 4, EDGE_MIN_GAMES = 200;
  function favMap() {
    const d = dash.data("favored"); const m = new Map();
    if (d.status === "ok") d.data.forEach((r) => m.set(r.role + "|" + r.champion + "|" + r.enemy, r));
    return m;
  }
  let FAV = new Map();
  function edgeOf(roles, mine, enemy) {
    for (const role of roles) {
      const r = FAV.get(role + "|" + mine + "|" + enemy); if (!r) continue;
      const e = Number(r.edge), n = Number(r.games);
      if (n < EDGE_MIN_GAMES || Math.abs(e) < EDGE_FAVORED) return null;
      return { who: e > 0 ? "you" : "enemy", stars: Math.abs(e) >= EDGE_HARD ? "★★" : "★", edge: e };
    }
    return null;
  }
  function edgeLabel(roles, mine, enemy) { const x = edgeOf(roles, mine, enemy); return x ? x.stars + " " + (x.who === "you" ? mine : enemy) : ""; }
  function gameEdge(g) {
    const bot = g.role === "ADC" || g.role === "Support";
    const parts = g.enemies ? g.enemies.split("|") : [];
    const same = bot ? (g.role === "ADC" ? parts[0] : parts[parts.length - 1]) : parts[0];
    const x = same ? edgeOf([g.role], g.champion, same) : null;
    const tag = (name) => (x && name === same && x.who === "enemy" ? name + " " + x.stars : name);
    return { mine: x && x.who === "you" ? g.champion + " " + x.stars : g.champion, vs: parts.map(tag).join(" + ") || g.vs };
  }

  function draw() {
    const p = P();
    ["side", "role"].forEach((k) => document.querySelectorAll(`#${k} button`).forEach((b) => b.setAttribute("aria-pressed", String(b.dataset.v === p[k]))));
    tiles(p); detail(p);
    dash.setLink(p.champ ? (p.side + "-" + p.champ.replace(/[^A-Za-z0-9]/g, "")) : "top");
  }

  dash.onData(() => {
    FAV = favMap();
    const fi = dash.data("favored_info");
    document.querySelectorAll('[data-source="favored_info"]').forEach((el) => { el.textContent = fi.status === "ok" ? String(fi.data[0][el.dataset.field]) : "—"; });
    const o = dash.data("overview");
    document.querySelectorAll('[data-source="overview"]').forEach((el) => { el.textContent = o.status === "ok" ? String(o.data[0][el.dataset.field]) : "—"; });
    draw();
  });
})();
</script>

## Step 4: build the dashboards

Processes whatever exports are in `DATA_DIR` and saves the tables to `DATA_DIR/dashboard_data/`. Re-run this and the two cells after it whenever you pull new games.

**Matchup stars:** upload the repo's `matchup_ratings.csv` to `DATA_DIR` (Files panel → upload). It rates every champion in each role from lolalytics.com (last 30 days, Emerald+; ★ at a 2+ point edge, ★★ at 4+). Without it the dashboards simply show no stars.

In [ ]:
import importlib, sys
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)
import lol_process, lol_render
importlib.reload(lol_process); importlib.reload(lol_render)

riot = lol_process.process_riot(DATA_DIR, TIMEZONE)
ranked = lol_process.process_opgg(DATA_DIR, GAME_NAME, TAG_LINE, TIMEZONE)
ratings = lol_process.load_ratings(DATA_DIR)

out_dir = os.path.join(DATA_DIR, "dashboard_data")
os.makedirs(out_dir, exist_ok=True)
for name, df in {**(riot or {}), "ranked_games": ranked}.items():
    if df is not None:
        df.to_csv(os.path.join(out_dir, f"{name}.csv"), index=False)

print("Riot export:", f"{len(riot['timeline_games'])} games" if riot else "not found (run Step 1)")
print("OP.GG export:", f"{len(ranked)} ranked games" if ranked is not None else "not found (Step 2 is optional)")
print("Matchup ratings:", f"{len(ratings)} matchups" if ratings is not None else "none")

In [ ]:
# ---- Ranked review ----
PLAYER = f"{GAME_NAME}#{TAG_LINE}"
R = lambda k: riot[k] if riot else None
notes = []
if ranked is None:
    notes.append("No OP.GG export found, so the rank, OP Score and session sections are hidden. Run Step 2 to add them.")
if riot is None:
    notes.append("No Riot export found, so the minute-by-minute, objectives and highlights sections are hidden. Run Step 1 to add them.")
if riot is None and ranked is None:
    print("Nothing to show yet: run Step 1 (and optionally Step 2) first.")
else:
    template = lol_render.trim_review(open("review_template.html").read(), ranked is not None, riot is not None)
    review_doc = lol_render.build_page(template, {
        "ranked_games": ranked, "timeline_games": R("timeline_games"), "timeline_deaths": R("timeline_deaths"),
        "timeline_minutes": R("timeline_minutes"), "timeline_fights": R("timeline_fights"), "objectives": R("objectives"),
    }, PLAYER, notes)
    path = os.path.join(DATA_DIR, "ranked_review.html")
    open(path, "w").write(review_doc)
    lol_render.show(review_doc, height=1200)
    print("Saved", path)

In [ ]:
# ---- Champions & matchups ----
if riot is None:
    print("The matchups dashboard needs the Riot export: run Step 1 first.")
else:
    template = open("matchups_template.html").read()
    if ratings is None:
        template = lol_render.without_ratings_note(template)
    matchups_doc = lol_render.build_page(template, {
        "champ_games": riot["champ_games"], "game_gold": riot["game_gold"], "game_events": riot["game_events"], "favored": ratings,
    }, PLAYER)
    path = os.path.join(DATA_DIR, "champion_matchups.html")
    open(path, "w").write(matchups_doc)
    lol_render.show(matchups_doc, height=1000)
    print("Saved", path)

Not endorsed by Riot Games. League of Legends is a trademark of Riot Games, Inc.